# KL-IG² — Distribution-Space IG²

**KL-IG²** lifts the IG² representation-descent idea from pixel space into
distribution space `(μ, logvar)`.

The path is built by descending the **expected representation distance**:

    L(μ, lv) = E_{x ~ N(μ, exp(lv)·I)} [ ‖φ(x) − φ(x_cf)‖² ]

where `φ` is a hidden-layer activation of the classifier and `x_cf` is a
counterfactual reference image.  Gradients w.r.t. `(μ, lv)` come from the
reparameterisation trick; updates use sign normalisation (fixed step size
regardless of gradient magnitude, matching the original IG² convention).

The resulting trajectory is consumed by the standard `KLIntegratedGradients`
integrator — **no integrator changes** compared to KL-IG.

What the `²` means structurally: each integration step accumulates

    (∂F_target/∂(μ,lv)) · (dμ, dlv)

where `dμ ≈ −lr_μ · sign(∂L_repdist/∂μ)`.  The displacement is the
counterfactual representation gradient.  Attribution is large at pixels where
both the explicand class gradient and the counterfactual contrast gradient
are large and aligned — the "²" (product of two gradient signals) in
distribution space.

**Verification protocol (5 checks) before trusting results:**
1. Loss trajectory — must be monotonically decreasing
2. Path length distribution — early-stopping rate
3. Visual path inspection — μ_k should morph toward the counterfactual class
4. Completeness check — Σ attr ≈ E[F(x)|explicand] − E[F(x)|cf]
5. lv-displacement check — lv must move non-trivially (not a horizontal line)


In [ ]:
# Local run: klig package lives in this directory (no git clone needed)
import os, sys
for _root in [os.getcwd(), '.', 'infocube-main']:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)


In [ ]:
import importlib, os, sys, math, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
if not hasattr(np, "trapz"): np.trapz = np.trapezoid  # NumPy>=2.0 compat
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Force-reload to avoid stale Colab module cache
import klig.core.rep_descent_path as _rdp_mod
importlib.reload(_rdp_mod)
import klig.core.ig2_integrator as _ig2_mod
importlib.reload(_ig2_mod)
import klig as _klig_mod
importlib.reload(_klig_mod)

from klig import (KLIntegratedGradients, AttributionResult,
                  RepDescentPath, make_phi_from_layer,
                  KLIGSquared, KLIGSquaredResult)
from klig.image.stopping import find_sigma_stop
from klig.core.path import LinearPath
from torchvision.models import resnet50, ResNet50_Weights
print('imports OK')


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
N_IMGS         = 10
SIGMA_FINAL    = 0.25             # σ for KL-IG linear / KL-IG² (fixed)

# RepDescentPath (path-build) hyperparameters
T_DESCENT      = 50               # max descent steps
LR_MU          = 0.05             # sign-normalised step in pixel space
LR_LV          = 0.10             # sign-normalised step in log-variance space
N_MC_DESCENT   = 16               # MC samples per descent step
LOSS_STOP      = 1e-3             # early-stop on descent loss
LV_FLOOR       = 2 * math.log(1 / 256)   # ≈ −11.09  (minimum σ clamp for descent)
LV_CEIL        = 4.0              # σ ≤ ~7.4
# ImageNet-normalised pixel range (3σ rule for each channel):
MU_MIN, MU_MAX = -2.64, 2.64

# KLIntegratedGradients (integration) hyperparameters
N_STEPS_INT    = 50               # quadrature points
N_MC_INT       = 10               # MC samples per integration step

# Evaluation
N_INSERTION_STEPS = 50
VIS_IMG_IDX    = 0
FORCE_RECOMPUTE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

# Save caches locally (next to the notebook), not Google Drive
CACHE_DIR = Path('klig2_val_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    'KLIG-Adaptive',
    'KL-IG (linear)',
    'KL-IG²',
    'KL-IG² (adaptive)',
    ]
COLORS = {
    'KLIG-Adaptive':   '#2d6a2d',
    'KL-IG (linear)':  '#333333',
    'KL-IG²':          '#e41a1c',
    'KL-IG² (adaptive)': '#8b0000',
}

In [ ]:
# ── Model + representation extractor ─────────────────────────────────────────
weights  = ResNet50_Weights.IMAGENET1K_V2
model    = resnet50(weights=weights).to(DEVICE).eval()
preprocess   = weights.transforms()
imagenet_labels = weights.meta['categories']

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
def denormalize(x): return x.cpu() * _STD + _MEAN

def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0)
    return a.gather(0, idx.unsqueeze(0)).squeeze(0)

# φ = output of layer4 (spatial feature map, 2048×7×7 for 224×224 inputs).
# Flattened inside RepDescentPath to a vector for distance computation.
phi = make_phi_from_layer(model, 'layer4')
print('φ layer: layer4  (ResNet50 last spatial block)')
print('model loaded; ResNet50 ready')

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
_cache_ds = CACHE_DIR / 'dataset.pkl'
if not FORCE_RECOMPUTE and _cache_ds.exists():
    with open(_cache_ds, 'rb') as f: dataset = pickle.load(f)
    print(f'[cache] dataset n={len(dataset)}')
else:
    from datasets import load_dataset as _hf
    _ds = _hf('evanarlian/imagenet_1k_resized_256', split='val', streaming=True)
    _ds = _ds.shuffle(seed=42, buffer_size=5000)   # <── mix classes before sampling
    dataset = []
    for item in tqdm(_ds.take(N_IMGS * 8), desc='loading'):
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            tgt  = int(logits.argmax(-1).item())
            conf = logits.softmax(-1)[0, tgt].item()
        if conf > 0.3:
            dataset.append({'x': x, 'target': tgt, 'idx': len(dataset)})
        if len(dataset) >= N_IMGS: break
    with open(_cache_ds, 'wb') as f: pickle.dump(dataset, f)
    print(f'Collected {len(dataset)} images')

In [ ]:
import os
for f in ['cf_y2_pool.pkl', 'klig2_dist_attrs.pkl']:
    p = CACHE_DIR / f
    if p.exists():
        os.remove(p); print(f'removed {f}')
    else:
        print(f'(absent) {f}')

In [ ]:
# ── Counterfactual selection: target second-most-likely class ────────────────
_cache_cf = CACHE_DIR / 'cf_y2_pool.pkl'

# (1) y_2 for every explicand
y2_per_idx = []
with torch.no_grad():
    for row in dataset:
        x = row['x'].to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top2  = probs.topk(2).indices.tolist()
        y2    = top2[1] if top2[0] == row['target'] else top2[0]
        y2_per_idx.append(int(y2))

needed_classes = set(y2_per_idx)
print(f'distinct y_2 classes needed: {len(needed_classes)}')

# (2) Build a pool — best-by-probability per needed class (handles rare classes)
CF_POOL_MAX_SCAN = 4000
CF_ACCEPT_PROB   = 0.30
if not FORCE_RECOMPUTE and _cache_cf.exists():
    with open(_cache_cf, 'rb') as f: cf_pool_cpu = pickle.load(f)
    print(f'[cache] cf_pool loaded ({len(cf_pool_cpu)} classes)')
    if len(set(cf_pool_cpu) & needed_classes) < len(needed_classes):
        print('  cache incomplete — rebuilding'); _cache_cf.unlink()
if FORCE_RECOMPUTE or not _cache_cf.exists():
    from datasets import load_dataset as _hf
    _ds = _hf('evanarlian/imagenet_1k_resized_256', split='val',
              streaming=True).shuffle(seed=0, buffer_size=5000)
    need = list(needed_classes)
    best, locked, scanned = {c: (-1.0, None) for c in need}, set(), 0
    pbar = tqdm(total=len(need), desc='cf pool')
    for item in _ds:
        scanned += 1
        if len(locked) >= len(need) or scanned >= CF_POOL_MAX_SCAN:
            break
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = model(x).softmax(-1)[0].cpu()
        x_cpu = x.cpu()
        for cc in need:
            if cc in locked: continue
            pp = float(probs[cc])
            if pp > best[cc][0]:
                best[cc] = (pp, x_cpu)
                if pp >= CF_ACCEPT_PROB:
                    locked.add(cc); pbar.update(1)
    pbar.close()
    cf_pool_cpu = {cc: best[cc][1] for cc in need if best[cc][1] is not None}
    with open(_cache_cf, 'wb') as f: pickle.dump(cf_pool_cpu, f)
    print(f'cf_pool built: {len(cf_pool_cpu)}/{len(need)} classes (scanned {scanned})')

cf_pool = {k: v.to(DEVICE) for k, v in cf_pool_cpu.items()}

# (3) Accessor — exact y_2 match, with safe fallback
_missing_classes = needed_classes - set(cf_pool.keys())
if _missing_classes:
    print(f'WARNING: {len(_missing_classes)} y_2 classes had no cf image found\n'
          f'  → falling back to any-other-class image for those explicands')

def pick_cf_image(idx):
    y2 = y2_per_idx[idx]
    if y2 in cf_pool:
        return cf_pool[y2]
    own_tgt = dataset[idx]['target']
    for k, v in cf_pool.items():
        if k != own_tgt: return v
    raise RuntimeError('cf_pool empty — cannot pick counterfactual')

# Demo
for i in [0, 1, 2]:
    pred_lbl = imagenet_labels[dataset[i]['target']]
    y2_lbl   = imagenet_labels[y2_per_idx[i]]
    matched  = '✓' if y2_per_idx[i] in cf_pool else '(fallback)'
    print(f'img {i}: predicted={pred_lbl!r:30s}  →  y_2={y2_lbl!r:30s}  {matched}')

In [ ]:
# ── Attribution loop ─────────────────────────────────────────────────────────
_cache_attr = CACHE_DIR / 'klig2_dist_attrs.pkl'

if not FORCE_RECOMPUTE and _cache_attr.exists():
    with open(_cache_attr, 'rb') as f:
        all_attrs, all_path_meta = pickle.load(f)
    print('[cache] attrs loaded')
else:
    all_attrs     = {m: [] for m in METHODS}
    all_path_meta = []

    ig_linear = KLIntegratedGradients(
        model, n_steps=N_STEPS_INT, n_samples=N_MC_INT,
        sigma_final=SIGMA_FINAL, device=DEVICE)

    for row in tqdm(dataset, desc='attributing'):
        x, tgt = row['x'], row['target']
        x1    = x.squeeze(0).to(DEVICE)
        x_cf  = pick_cf_image(row['idx']).squeeze(0).to(DEVICE)
        meta  = {}

        # shared adaptive sigma for both adaptive methods
        sig_adapt = find_sigma_stop(model, x, tgt, tau=0.95, n_samples=32, n_iter=12)

        # 1) KLIG-Adaptive: LinearPath + adaptive sigma
        r_adapt = KLIntegratedGradients(
            model, n_steps=N_STEPS_INT, n_samples=N_MC_INT,
            sigma_final=sig_adapt, path=LinearPath(), device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['KLIG-Adaptive'].append(absmax_collapse(r_adapt.attr).cpu())

        # 2) KL-IG linear (parametric baseline)
        r0 = ig_linear.attribute(x1, target=tgt)
        all_attrs['KL-IG (linear)'].append(absmax_collapse(r0.attr).cpu())

        # 3) KL-IG²: KLIGSquared — forward IG² integration, model-derived baseline
        ig2 = KLIGSquared(
            model, phi, x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc_path=N_MC_DESCENT, n_mc_grad=N_MC_INT,
            sigma_start=SIGMA_FINAL, loss_stop=LOSS_STOP,
            lv_floor=LV_FLOOR, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE)
        r1 = ig2.attribute(x1, target=tgt)
        all_attrs['KL-IG²'].append(absmax_collapse(r1.attr_mu).cpu())
        meta['loss_traj'] = list(r1.loss_trajectory)
        meta['path_len']  = len(r1.traj_mu) - 1
        meta['traj_mu']   = [m.cpu() for m in r1.traj_mu]
        meta['traj_lv']   = [v.cpu() for v in r1.traj_lv]

        all_path_meta.append(meta)

        # 4) KL-IG² (adaptive): KLIGSquared with adaptive sigma_start
        lv_floor_adapt = 2 * math.log(sig_adapt)
        r2 = KLIGSquared(
            model, phi, x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc_path=N_MC_DESCENT, n_mc_grad=N_MC_INT,
            sigma_start=sig_adapt, loss_stop=LOSS_STOP,
            lv_floor=lv_floor_adapt, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE
        ).attribute(x1, target=tgt)
        all_attrs['KL-IG² (adaptive)'].append(absmax_collapse(r2.attr_mu).cpu())

    with open(_cache_attr, 'wb') as f:
        pickle.dump((all_attrs, all_path_meta), f)
    print('Done.')

#Verify

In [ ]:
# ── Verification 1: loss trajectory ──────────────────────────────────────────
N_SHOW = 8
fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
for i in range(min(N_SHOW, len(all_path_meta))):
    ax.plot(all_path_meta[i]['loss_traj'],
             alpha=0.6, lw=1, color=COLORS['KL-IG²'])
ax.axhline(LOSS_STOP, color='red', lw=1, ls='--', label=f'loss_stop={LOSS_STOP}')
ax.set_xlabel('Descent step'); ax.set_ylabel('E[‖φ(x) − φ(x_cf)‖²]')
ax.set_title('Verification 1: descent loss — must be monotone ↓')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

n_violate = sum(
    any(b > a + 1e-6 for a, b in zip(m['loss_traj'], m['loss_traj'][1:]))
    for m in all_path_meta
)
print(f'Non-monotone paths: {n_violate}/{len(all_path_meta)}  '
      f'(>0 → lr_mu or lr_lv too large)')


In [ ]:
# ── Verification 4: completeness ─────────────────────────────────────────────
N_CC      = 20
N_MC_CC   = 64

def mc_expected_f(model, mu, lv_scalar, target, n=N_MC_CC):
    with torch.no_grad():
        std = math.exp(0.5 * lv_scalar)
        z   = mu.unsqueeze(0) + std * torch.randn(n, *mu.shape, device=mu.device)
        return float(model(z).softmax(-1)[:, target].mean().item())

rows_cc = []
for i, row in enumerate(tqdm(dataset[:N_CC], desc='completeness')):
    x, tgt = row['x'], row['target']
    x1      = x.squeeze(0).to(DEVICE)
    x_cf_i  = pick_cf_image(row['idx']).squeeze(0).to(DEVICE)
    meta    = all_path_meta[i]

    # E[F | explicand Gaussian] at s=1
    f_expl = mc_expected_f(model, x1,     LV_FLOOR, tgt)
    # E[F | cf end of path] at s=0: use the first waypoint from the stored trajectory
    mu_s0  = meta['traj_mu'][0].to(DEVICE)
    lv_s0  = meta['traj_lv'][0].to(DEVICE)
    lv_s0_scalar = float(lv_s0.mean().item())
    f_cf   = mc_expected_f(model, mu_s0, lv_s0_scalar, tgt)

    delta_f   = f_expl - f_cf
    sum_attr  = float(all_attrs['KL-IG²'][i].sum().item())
    rows_cc.append({'delta_f': delta_f, 'sum_attr': sum_attr,
                    'err': abs(sum_attr - delta_f),
                    'rel': abs(sum_attr - delta_f) / (abs(delta_f) + 1e-8) * 100})

err   = np.array([r['err']  for r in rows_cc])
rel   = np.array([r['rel']  for r in rows_cc])
passed = (rel < 5.0).mean() * 100

print(f'Completeness check  (n={N_CC})')
print(f'  mean |err|:  {err.mean():.4f}')
print(f'  mean rel%:   {rel.mean():.2f}%')
print(f'  pass (<5%):  {passed:.1f}%')
if passed < 50:
    print('  WARNING: <50% pass — integration may be buggy; '
          'check n_steps, n_mc_int, or step size.')

fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor='white')
delta_fs  = [r['delta_f']  for r in rows_cc]
sum_attrs = [r['sum_attr'] for r in rows_cc]
axes[0].scatter(delta_fs, sum_attrs, alpha=0.7, color=COLORS['KL-IG²'], s=30)
lo, hi = min(delta_fs + sum_attrs), max(delta_fs + sum_attrs)
axes[0].plot([lo, hi], [lo, hi], 'k--', lw=1, label='perfect')
axes[0].set_xlabel('E[F|explicand] − E[F|cf]'); axes[0].set_ylabel('Σ attr')
axes[0].set_title('Completeness scatter'); axes[0].legend()
axes[1].hist(rel, bins=20, color=COLORS['KL-IG²'], alpha=0.8)
axes[1].axvline(5, color='red', lw=1.5, ls='--', label='5% threshold')
axes[1].set_xlabel('Relative error %'); axes[1].set_ylabel('Count')
axes[1].set_title('Relative completeness error'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Verification 5: lv-displacement (path taxonomy figure) ──────────────────
def _norm(v):
    v = np.asarray(v, dtype=float)
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo + 1e-12)

N_SHOW_LV = min(12, len(all_path_meta))
fig, ax   = plt.subplots(figsize=(7, 6), facecolor='white')

lv_disps, mu_disps = [], []
for i in range(N_SHOW_LV):
    meta = all_path_meta[i]
    mu_seq = np.stack([m.flatten().numpy() for m in meta['traj_mu']])
    lv_seq = np.stack([v.flatten().numpy() for v in meta['traj_lv']])
    mu_d   = _norm(np.linalg.norm(mu_seq - mu_seq[0], axis=1))
    lv_d   = _norm(np.linalg.norm(lv_seq - lv_seq[0], axis=1))
    mu_disps.append(float(mu_d.max())); lv_disps.append(float(lv_d.max()))
    ax.plot(mu_d, lv_d, alpha=0.55, lw=1.5, color=COLORS['KL-IG²'])

# horizontal line at lv=0 marks the failure case (pure pixel descent)
ref_mu = np.linspace(0, 1, 100)
ax.plot(ref_mu, np.zeros(100), 'k--', lw=1, alpha=0.35,
         label='failure: lv never moves')

ax.set_xlabel('μ-displacement (norm.)', fontsize=11)
ax.set_ylabel('lv-displacement (norm.)', fontsize=11)
ax.set_title('Verification 5: path geometry — lv must move non-trivially\n'
             '(horizontal = pixel-space IG² reimplementation)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

mean_lv_disp = np.mean(lv_disps)
print(f'Mean peak lv-displacement (norm.): {mean_lv_disp:.3f}')
if mean_lv_disp < 0.05:
    print('  WARNING: lv barely moves — try increasing LR_LV or reducing LOSS_STOP')
else:
    print('  OK: lv axis is contributing to the path geometry')

#Metrics

## Insertion / Deletion AUC

In [ ]:
# ── Insertion / Deletion AUC ─────────────────────────────────────────────────
_cache_id = CACHE_DIR / 'klig2_dist_ins_del.pkl'


def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]

    order = attr_map.detach().view(-1).abs().argsort(descending=True)
    pps = max(1, H * W // n_steps)

    blur = F.avg_pool2d(x, kernel_size=31, stride=1, padding=15)

    x_ins = blur.clone()
    x_del = x.clone()

    ins_s, del_s = [], []

    with torch.no_grad():
        for step in range(n_steps):
            pix = order[step * pps:(step + 1) * pps]

            for ch in range(C):
                x_ins[:, ch].reshape(-1)[pix] = x[:, ch].reshape(-1)[pix]
                x_del[:, ch].reshape(-1)[pix] = blur[:, ch].reshape(-1)[pix]

            ins_s.append(model(x_ins).softmax(-1)[0, target].item())
            del_s.append(model(x_del).softmax(-1)[0, target].item())

    return (
        float(np.trapz(ins_s) / n_steps),
        float(np.trapz(del_s) / n_steps)
    )


if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, 'rb') as f:
        ins_auc, del_auc = pickle.load(f)
    print('[cache] insertion/deletion loaded')

else:
    ins_auc = defaultdict(list)
    del_auc = defaultdict(list)

    for row in tqdm(dataset, desc='ins/del'):
        x, tgt = row['x'], row['target']

        for m in METHODS:
            attr = all_attrs[m][row['idx']].to(DEVICE).unsqueeze(0)

            i_, d_ = insertion_deletion(model, x, attr, tgt)

            ins_auc[m].append(i_)
            del_auc[m].append(d_)

    ins_auc = dict(ins_auc)
    del_auc = dict(del_auc)

    with open(_cache_id, 'wb') as f:
        pickle.dump((ins_auc, del_auc), f)

    print('[cache] insertion/deletion saved')


fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor='white')

for ax, (title, aucs) in zip(
    axes,
    [
        ('Insertion AUC ↑', ins_auc),
        ('Deletion AUC ↓', del_auc)
    ]
):
    for xi, m in enumerate(METHODS):
        v = aucs[m]
        mu_ = np.mean(v)
        ci = 1.96 * np.std(v) / len(v) ** 0.5

        ax.bar(
            xi,
            mu_,
            color=COLORS[m],
            alpha=0.85,
            width=0.6
        )

        ax.errorbar(
            xi,
            mu_,
            yerr=ci,
            fmt='none',
            color='black',
            capsize=4
        )

    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
    ax.set_title(title)

plt.suptitle(f'Insertion / Deletion AUC  (n={N_IMGS})', fontsize=11)
plt.tight_layout()
plt.show()

## Sensitivity-n (baseline-matched)

For `KL-IG² (dist)` the path starts at the counterfactual image, so sensitivity-n masks to `x_cf`, not zero.

In [ ]:

    # ── Sensitivity-n (baseline-matched) ──────────────────────────────────────────
    _cache_sn = CACHE_DIR / 'klig2_dist_sens_n.pkl'

    def sensitivity_n(model, x, attr_map, target, baseline,
                      n_subsets=50, subset_size=0.1):
        rng_sn    = np.random.default_rng(42)
        attr_flat = attr_map.cpu().detach().view(-1).numpy()
        n_pix     = attr_flat.size
        n_sel     = max(1, int(n_pix * subset_size))
        df_list, da_list = [], []
        with torch.no_grad():
            f_x = model(x).softmax(-1)[0, target].item()
            for _ in range(n_subsets):
                idx    = rng_sn.choice(n_pix, n_sel, replace=False)
                x_mask = x.clone()
                for ch in range(x.shape[1]):
                    x_mask[:, ch].reshape(-1)[idx] = baseline[:, ch].reshape(-1)[idx]
                f_mask = model(x_mask).softmax(-1)[0, target].item()
                df_list.append(f_x - f_mask)
                da_list.append(float(attr_flat[idx].sum()))
        df, da = np.array(df_list), np.array(da_list)
        if df.std() < 1e-9 or da.std() < 1e-9: return 0.0
        return float(np.corrcoef(df, da)[0, 1])

    if not FORCE_RECOMPUTE and _cache_sn.exists():
        with open(_cache_sn, 'rb') as f: sens_n = pickle.load(f)
    else:
        sens_n  = defaultdict(list)
        zeros   = None
        for row in tqdm(dataset, desc='sensitivity-n'):
            x, tgt = row['x'], row['target']
            if zeros is None: zeros = torch.zeros_like(x)
            x_cf_i  = pick_cf_image(row['idx']).to(DEVICE)
            x_cf_i_4d = x_cf_i if x_cf_i.dim() == 4 else x_cf_i.unsqueeze(0)
            baselines = {m: zeros for m in METHODS}
            baselines['KL-IG²'] = x_cf_i_4d
            baselines['KL-IG² (adaptive)'] = x_cf_i_4d
            for m in METHODS:
                if m not in all_attrs or row['idx'] >= len(all_attrs[m]): continue
                attr = all_attrs[m][row['idx']].to(DEVICE).unsqueeze(0)
                sens_n[m].append(sensitivity_n(model, x, attr, tgt, baselines[m]))
        sens_n = dict(sens_n)
        with open(_cache_sn, 'wb') as f: pickle.dump(sens_n, f)

    fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
    for xi, m in enumerate(METHODS):
        v = sens_n[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
        ax.bar(xi, mu_, color=COLORS[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='black', capsize=4)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_ylabel('Pearson r (signed)')
    ax.set_title(f'Sensitivity-n (baseline-matched,  n={len(dataset)})')
    plt.tight_layout(); plt.show()


## Sparsity (Gini)

In [ ]:

    # ── Sparsity (Gini) ──────────────────────────────────────────────────────────
    def gini(v):
        v = v.abs().flatten().numpy(); v = np.sort(v); n = len(v)
        return float((2*np.arange(1,n+1) - n - 1) @ v / (n * v.sum() + 1e-12))

    gini_scores = {m: [gini(all_attrs[m][i]) for i in range(len(dataset))]
                   for m in METHODS}

    fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
    for xi, m in enumerate(METHODS):
        v = gini_scores[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
        ax.bar(xi, mu_, color=COLORS[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='black', capsize=4)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Gini ↑'); ax.set_title(f'Sparsity  (n={len(dataset)})')
    plt.tight_layout(); plt.show()


## OFR (Object Focus Ratio)

Measures what fraction of the total attribution mass lands **inside the object region**.
The object mask is estimated via GrabCut seeded by the reference method's (KL-IG linear)
attribution map.  A higher OFR means the method concentrates its attributions on the
object rather than on background pixels.

    OFR = Σ |attr[mask]| / Σ |attr|

In [ ]:

    # ── OFR (Object Focus Ratio) ──────────────────────────────────────────────────
    import cv2

    _cache_ofr = CACHE_DIR / 'klig2_dist_ofr.pkl'

    def estimate_object_mask(x, attr_map):
        """GrabCut-based object mask seeded by attribution percentiles."""
        H, W = attr_map.shape
        a    = np.abs(attr_map.detach().cpu().numpy())
        seed = (a >= np.percentile(a, 80)).astype(np.uint8)
        img_rgb = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
        img_bgr = (img_rgb * 255).clip(0, 255).astype(np.uint8)[:, :, ::-1].copy()
        gc_mask = np.where(seed, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)
        gc_mask[(a >= np.percentile(a, 95))] = cv2.GC_FGD
        border  = max(H, W) // 10
        edge    = np.zeros((H, W), dtype=np.uint8)
        edge[:border, :] = 1; edge[-border:, :] = 1
        edge[:, :border] = 1; edge[:, -border:] = 1
        gc_mask[(edge == 1) & (a < np.percentile(a, 10))] = cv2.GC_BGD
        try:
            bgd = np.zeros((1, 65), np.float64)
            fgd = np.zeros((1, 65), np.float64)
            cv2.grabCut(img_bgr, gc_mask, None, bgd, fgd, 5, cv2.GC_INIT_WITH_MASK)
            return np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD),
                            1, 0).astype(np.uint8)
        except Exception:
            return seed

    def object_focus_ratio(attr_map, obj_mask):
        """Fraction of |attr| mass inside the object mask."""
        a = np.abs(attr_map.detach().cpu().numpy())
        total = a.sum()
        return float(a[obj_mask == 1].sum() / total) if total > 1e-12 else 0.0

    if not FORCE_RECOMPUTE and _cache_ofr.exists():
        with open(_cache_ofr, 'rb') as f: ofr_scores = pickle.load(f)
        print(f'[cache] OFR loaded')
    else:
        ofr_scores = {m: [] for m in METHODS}
        for row in tqdm(dataset, desc='OFR'):
            x, tgt = row['x'], row['target']
            # Reference mask: use KL-IG (linear) attribution as GrabCut seed
            ref_attr = all_attrs['KL-IG (linear)'][row['idx']].to(DEVICE)
            obj_mask = estimate_object_mask(x, ref_attr)
            for m in METHODS:
                attr = all_attrs[m][row['idx']]
                ofr_scores[m].append(object_focus_ratio(attr, obj_mask))
        with open(_cache_ofr, 'wb') as f: pickle.dump(ofr_scores, f)
        print('OFR computed and cached.')

    # ── Plot ──────────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5), facecolor='white')
    for xi, m in enumerate(METHODS):
        v   = ofr_scores[m]
        mu_ = np.mean(v)
        ci  = 1.96 * np.std(v) / len(v) ** 0.5
        ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6,
               edgecolor='white', linewidth=0.8)
        ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)

    ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0)
    ax.set_axisbelow(True)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=15, ha='right', fontsize=10)
    ax.set_ylabel('OFR ↑  (fraction of |attr| inside object)', fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'Object Focus Ratio  (n={len(dataset)})', fontsize=12)
    plt.tight_layout(); plt.show()

    for m in METHODS:
        v = ofr_scores[m]
        print(f'{m:25s}  mean={np.mean(v):.3f}  ci95=±{1.96*np.std(v)/len(v)**0.5:.3f}')


In [ ]:
# ── Master Metrics Table ──────────────────────────────────────────────────────
# Collects every metric into one DataFrame. Each source is guarded with
# globals().get(...) so missing metrics (commented-out cells) just show '—'.
import pandas as pd
from scipy.stats import spearmanr as _sr

ci95 = lambda v: 1.96 * np.std(v) / len(v) ** 0.5
fmt  = lambda v: f'{np.mean(v):.3f}±{ci95(v):.3f}' if len(v) else '—'

# Metric dicts (may be undefined if their cells are commented out)
_gini = globals().get('gini_scores', {})
_ins  = globals().get('ins_auc',     {})
_del  = globals().get('del_auc',     {})
_sn   = globals().get('sens_n',      {})
_ofr  = globals().get('ofr_scores',  {})
# Class-sensitivity sources
_scat = globals().get('scatter_all', {})   # (clip_dist, cos_dist) per pair
_fa   = globals().get('fa_scores',   {})   # CASE feature-agreement
_res  = globals().get('results',     {})   # CASE Wilcoxon p-values

rows = []
for m in METHODS:
    r = {'Method': m}

    # ── Faithfulness / sparsity ──
    r['Gini ↑']    = fmt(_gini.get(m, []))
    r['Ins AUC ↑'] = fmt(_ins.get(m, []))
    r['Del AUC ↓'] = fmt(_del.get(m, []))
    r['Sens-n ↑']  = fmt(_sn.get(m, []))
    r['OFR ↑']     = fmt(_ofr.get(m, []))

    # ── Class sensitivity: |Spearman ρ| (CLIP dist vs cos-sim) ──
    pts = _scat.get(m, [])
    if len(pts) >= 5:
        rho_, _ = _sr([p[0] for p in pts], [p[1] for p in pts])
        r['|CS ρ| ↑'] = f'{abs(rho_):.3f}'
    else:
        r['|CS ρ| ↑'] = '—'

    # ── CASE Feature Agreement (median + one-sided Wilcoxon p) ──
    fa = np.array(_fa.get(m, []))
    r['CASE FA ↓'] = f'{np.median(fa):.3f}' if len(fa) else '—'
    r['CASE p']    = f'{_res[m]["p"]:.4f}' if m in _res else '—'

    rows.append(r)

df_all = pd.DataFrame(rows).set_index('Method')

# Drop all-empty columns (metrics never computed)
df_all = df_all.loc[:, (df_all != '—').any(axis=0)]

print(df_all.to_string())
print()
print("Arrows show the better direction:")
print("  ↑ higher is better : Gini, Ins AUC, Sens-n, OFR, |CS ρ|")
print("  ↓ lower  is better : Del AUC, CASE FA")
print()
print("Class-sensitivity metrics (2 independent protocols):")
print("  |CS ρ|   — |Spearman| of CLIP distance vs attribution cos-sim   (higher = sensitive)")
print("  CASE FA  — top-5% pixel overlap between top-1/top-2 maps         (lower  = sensitive)")
df_all

# Class Sens

In [40]:
# ── Class Sensitivity Scatter — Step 1: filter multi-class images ────────────
import itertools
import nltk; nltk.download('wordnet', quiet=True)
from nltk.corpus import wordnet as wn
from scipy.stats import spearmanr

CS_PROB_THRESH = 0.1
CS_MAX_IMGS    = 200
CS_MAX_SCAN    = 3000
NON_ANIMAL_CLS = set(range(400, 1000))

_cache_multi = CACHE_DIR / 'klig2_dist_multiprob.pkl'

def label_to_synset(label):
    phrase = label.lower().split(',')[0].strip()
    cands  = []
    for candidate in [phrase.replace(' ', '_'), phrase]:
        cands.extend(wn.synsets(candidate, pos=wn.NOUN))
    if not cands:
        for w in sorted(phrase.split(), key=len, reverse=True):
            if len(w) > 3:
                cands.extend(wn.synsets(w, pos=wn.NOUN))
                if cands: break
    if not cands: return None
    return max(cands, key=lambda s: s.min_depth())

cls_synsets = {i: label_to_synset(imagenet_labels[i]) for i in range(1000)}

# FIX: cosine_dist_cs is SIGNED (no np.clip) — must match Step 2 so cell-run-order
#      doesn't matter. Clipping zeroed negatives and destroyed KL-IG2's sign.
def cosine_dist_cs(a_i, a_j):
    ai = a_i.astype(np.float64).ravel()      # signed (no clip)
    aj = a_j.astype(np.float64).ravel()      # signed (no clip)
    denom = np.linalg.norm(ai) * np.linalg.norm(aj)
    if denom < 1e-12: return 1.0
    return float(1.0 - (ai @ aj) / denom)

if not FORCE_RECOMPUTE and _cache_multi.exists():
    multi_imgs = pickle.load(open(_cache_multi, 'rb'))
    print(f'[cache] {len(multi_imgs)} multi-class images')
else:
    from datasets import load_dataset as _hf
    _ds_cs = _hf('evanarlian/imagenet_1k_resized_256', split='val', streaming=True)
    _ds_cs = _ds_cs.shuffle(seed=42, buffer_size=5000)
    multi_imgs = []
    scanned = 0
    for item in tqdm(_ds_cs, desc='scanning for multi-class', total=CS_MAX_SCAN):
        if len(multi_imgs) >= CS_MAX_IMGS or scanned >= CS_MAX_SCAN:
            break
        scanned += 1
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = model(x).softmax(-1)[0].cpu()
        high = (probs > CS_PROB_THRESH).nonzero(as_tuple=True)[0].tolist()
        if len(high) < 2: continue
        if high[0] not in NON_ANIMAL_CLS: continue
        high = sorted(high, key=lambda c: probs[c].item(), reverse=True)
        sig_adapt = find_sigma_stop(model, x, high[0], tau=0.95,
                                    n_samples=32, n_iter=12)
        multi_imgs.append({
            'idx':        len(multi_imgs),
            'x':          x,
            'high_cls':   high,
            'high_probs': [probs[c].item() for c in high],
            'sig_adapt':  sig_adapt,
        })
    pickle.dump(multi_imgs, open(_cache_multi, 'wb'))
    print(f'Found {len(multi_imgs)} multi-class images '
          f'(scanned {scanned}, thresh={CS_PROB_THRESH})')

n_pairs_total = sum(
    len(list(itertools.combinations(d['high_cls'], 2))) for d in multi_imgs
)
print(f'Total class pairs: {n_pairs_total}')

scanning for multi-class:  36%|███▌      | 1066/3000 [01:25<02:35, 12.47it/s]

Found 200 multi-class images (scanned 1066, thresh=0.1)
Total class pairs: 233


In [46]:
FORCE_RECOMPUTE = True

In [47]:
# ===== Class-sensitivity scatter — Wu-Palmer ImageNet-tree distance (no CLIP) =====
#  x-axis: 1 - Wu-Palmer similarity (WordNet taxonomy distance) between y1,y2
#  y-axis: SIGNED cosine distance between attribution maps
#  Fixes kept: signed cosine · sum-collapse · complete KL-IG2 (mu+lv) · SHARED lv_floor
#  Needs cls_synsets from Step 1 (WordNet synset per class).
import nltk; nltk.download('wordnet', quiet=True)
_cache_scatter   = CACHE_DIR / 'klig2_cs_scatter_wup.pkl'
_cache_scatter_n = CACHE_DIR / 'klig2_cs_scatter_wup_n.pkl'

# ── Wu-Palmer tree distance (sole class-distance axis) ───────────────────────
def wup_tree_dist(ci, cj):
    si, sj = cls_synsets[ci], cls_synsets[cj]
    if si is None or sj is None:
        return None
    sim = si.wup_similarity(sj)
    if sim is None or sim <= 0:
        return None
    return 1.0 - sim                      # DISTANCE: higher = taxonomically farther

# ── SIGNED cosine distance ───────────────────────────────────────────────────
def cosine_dist_cs(a_i, a_j):
    ai = a_i.astype(np.float64).ravel(); aj = a_j.astype(np.float64).ravel()
    denom = np.linalg.norm(ai) * np.linalg.norm(aj)
    return 1.0 if denom < 1e-12 else float(1.0 - (ai @ aj) / denom)

# ── Bin selection by tree distance (Wu-Palmer is chunky → coarse bins) ───────
TREE_BINS = [(0.0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.01)]
MAX_PER_BIN = 30
_buckets = {i: [] for i in range(len(TREE_BINS))}
for _d in multi_imgs:
    if not _d.get('high_cls') or len(_d['high_cls']) < 2:
        continue
    td = wup_tree_dist(_d['high_cls'][0], _d['high_cls'][1])
    if td is None:
        continue
    b = next((i for i,(lo,hi) in enumerate(TREE_BINS) if lo <= td < hi), None)
    if b is not None and len(_buckets[b]) < MAX_PER_BIN:
        _buckets[b].append(_d)
multi_imgs_scatter = [d for bk in _buckets.values() for d in bk]
print(f"multi_imgs_scatter: {len(multi_imgs_scatter)} images, "
      f"{sum(1 for b in _buckets.values() if b)} tree bins")
for _bi,(lo,hi) in enumerate(TREE_BINS):
    print(f"  [{lo:.2f},{hi:.2f}): {len(_buckets[_bi])}")

# ── CF pool for the scatter's own y2 classes (best-by-probability) ───────────
_cache_cf_scatter = CACHE_DIR / 'klig2_cf_scatter_pool.pkl'
_need_y2 = {d['high_cls'][1] for d in multi_imgs_scatter if len(d['high_cls']) > 1}
print(f'Scatter needs CF images for {len(_need_y2)} distinct y2 classes')
if not FORCE_RECOMPUTE and _cache_cf_scatter.exists():
    cf_scatter_cpu = pickle.load(open(_cache_cf_scatter, 'rb'))
    if len(set(cf_scatter_cpu) & _need_y2) < len(_need_y2):
        print('  cached scatter-CF pool incomplete — rebuilding'); _cache_cf_scatter.unlink()
if FORCE_RECOMPUTE or not _cache_cf_scatter.exists():
    cf_scatter_cpu = {c: cf_pool[c].cpu() for c in _need_y2 if 'cf_pool' in globals() and c in cf_pool}
    _still = _need_y2 - set(cf_scatter_cpu)
    if _still:
        from datasets import load_dataset as _hf
        _s = _hf('evanarlian/imagenet_1k_resized_256', split='val', streaming=True).shuffle(seed=11, buffer_size=5000)
        best, locked = {c: (-1.0, None) for c in _still}, set()
        scanned, CF_SCAN_MAX, ACCEPT = 0, 20000, 0.30
        pbar = tqdm(total=len(_still), desc='scatter CF pool')
        for item in _s:
            scanned += 1
            if len(locked) >= len(_still) or scanned >= CF_SCAN_MAX: break
            im = item['image']
            if im.mode != 'RGB': im = im.convert('RGB')
            xx = preprocess(im).unsqueeze(0).to(DEVICE)
            with torch.no_grad(): probs = model(xx).softmax(-1)[0].cpu()
            xx_cpu = xx.cpu()
            for c in _still:
                if c in locked: continue
                p = float(probs[c])
                if p > best[c][0]:
                    best[c] = (p, xx_cpu)
                    if p >= ACCEPT: locked.add(c); pbar.update(1)
        pbar.close()
        for c in _still:
            if best[c][1] is not None: cf_scatter_cpu[c] = best[c][1]
    pickle.dump(cf_scatter_cpu, open(_cache_cf_scatter, 'wb'))
    print(f'scatter-CF pool: {len(cf_scatter_cpu)}/{len(_need_y2)} classes')
cf_scatter_pool = {c: v.to(DEVICE) for c, v in cf_scatter_cpu.items()}

# ── Cheap settings + method machinery (unchanged) ────────────────────────────
N_STEPS_SCATTER, N_MC_SCATTER, T_DESCENT_SCATTER, N_MC_DESC_SCATTER = 25, 3, 25, 8
ig_linear_cs2 = KLIntegratedGradients(model, n_steps=N_STEPS_SCATTER, n_samples=N_MC_SCATTER,
                                      sigma_final=SIGMA_FINAL, device=DEVICE)
def _pick_cf_scatter(d):
    y2 = d['high_cls'][1] if len(d['high_cls']) > 1 else d['high_cls'][0]
    if y2 in cf_scatter_pool: return cf_scatter_pool[y2]
    if 'cf_pool' in globals() and y2 in cf_pool: return cf_pool[y2]
    return next(iter(cf_scatter_pool.values()))
def _build_klig2(x_cf_img, sigma_start, lv_floor):
    return KLIGSquared(model, phi, x_cf_img, T=T_DESCENT_SCATTER, lr_mu=LR_MU, lr_lv=LR_LV,
        n_mc_path=N_MC_DESC_SCATTER, n_mc_grad=N_MC_SCATTER, sigma_start=sigma_start,
        loss_stop=LOSS_STOP, lv_floor=lv_floor, lv_ceil=LV_CEIL,
        mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE)
def _build_gradpath_once(klig2_obj, x1):
    x1d = x1.to(DEVICE); x1d = x1d.squeeze(0) if (x1d.dim()>1 and x1d.shape[0]==1) else x1d
    tm, tl, _ = klig2_obj._build_gradpath(x1d, x1d.shape); return tm, tl
def _klig2_integrate_only(klig2_obj, x1, target, traj_mu, traj_lv):
    # complete KL-IG2 = mu + lv, integrated on the (shared-floor) gradpath
    x1d = x1.to(DEVICE); x1d = x1d.squeeze(0) if (x1d.dim()>1 and x1d.shape[0]==1) else x1d
    _, obj = klig2_obj._resolve_target(x1d, int(target))
    saved = [p.requires_grad for p in klig2_obj.model.parameters()]
    for p in klig2_obj.model.parameters(): p.requires_grad_(False)
    acc = torch.zeros_like(x1d)
    try:
        for k in range(len(traj_mu)-1):
            g_mu, g_lv = klig2_obj._eval_gradients(traj_mu[k], traj_lv[k], x1d.shape, obj)
            with torch.no_grad():
                acc.add_(g_mu * (traj_mu[k]-traj_mu[k+1]) + g_lv * (traj_lv[k]-traj_lv[k+1]))
    finally:
        for p, s in zip(klig2_obj.model.parameters(), saved): p.requires_grad_(s)
    return acc
def _attr_for_class_fast(m, x1, c, klig2_fixed=None, klig2_adapt=None,
                         path_fixed=None, path_adapt=None, sig_adapt=None):
    if m == 'KLIG-Adaptive':
        attr = KLIntegratedGradients(model, n_steps=N_STEPS_SCATTER, n_samples=N_MC_SCATTER,
                 sigma_final=(sig_adapt or SIGMA_FINAL), path=LinearPath(), device=DEVICE
               ).attribute(x1, target=int(c)).attr
    elif m == 'KL-IG (linear)':
        attr = ig_linear_cs2.attribute(x1, target=int(c)).attr
    elif m == 'KL-IG²':
        attr = _klig2_integrate_only(klig2_fixed, x1, int(c), path_fixed[0], path_fixed[1])
    elif m == 'KL-IG² (adaptive)':
        attr = _klig2_integrate_only(klig2_adapt, x1, int(c), path_adapt[0], path_adapt[1])
    else:
        raise ValueError(m)
    a = attr.squeeze(0) if attr.dim() == 4 else attr
    return a.sum(0).detach().cpu().numpy().ravel()        # signed sum-collapse

# ── Invalidate stale cache + run ─────────────────────────────────────────────
for _c in (_cache_scatter, _cache_scatter_n):
    if _c.exists(): _c.unlink()
scatter_all = {m: [] for m in METHODS}

for d in tqdm(multi_imgs_scatter, desc='class-sens scatter (Wu-Palmer)'):
    x1       = d['x'].squeeze(0).to(DEVICE)
    high_cls = d['high_cls']
    sig_per_cls = dict(d.get('sig_per_cls', {}))
    x_cf_img = _pick_cf_scatter(d); x_cf_img = (x_cf_img.squeeze(0) if x_cf_img.dim()==4 else x_cf_img).to(DEVICE)

    def _sigma_for(cls):
        s = sig_per_cls.get(cls)
        if s is None:
            s = find_sigma_stop(model, x1, int(cls), tau=0.95, n_samples=32, n_iter=12)
            sig_per_cls[cls] = s
        return s

    # SHARED lv_floor across the pair (Top-1's floor) — the Option-B fix
    _lv_floor_sh = 2 * math.log(_sigma_for(high_cls[0]))

    klig2_fixed = path_fixed = None
    if 'KL-IG²' in METHODS:
        klig2_fixed = _build_klig2(x_cf_img, SIGMA_FINAL, LV_FLOOR)
        path_fixed  = _build_gradpath_once(klig2_fixed, x1)

    _adapt_cache = {}
    def _adapt_for(cls):
        if cls not in _adapt_cache:
            sig_c = _sigma_for(cls)                              # per-class sigma_start
            k2    = _build_klig2(x_cf_img, sig_c, _lv_floor_sh)  # SHARED lv_floor
            _adapt_cache[cls] = (k2, _build_gradpath_once(k2, x1), sig_c)
        return _adapt_cache[cls]

    attr_cache = {}
    for m in METHODS:
        for cls in high_cls:
            if (m, cls) in attr_cache: continue
            if m == 'KL-IG² (adaptive)':
                k2, pth, sg = _adapt_for(cls)
                attr_cache[(m,cls)] = _attr_for_class_fast(m, x1, cls, klig2_adapt=k2, path_adapt=pth, sig_adapt=sg)
            elif m == 'KLIG-Adaptive':
                attr_cache[(m,cls)] = _attr_for_class_fast(m, x1, cls, sig_adapt=_sigma_for(cls))
            elif m == 'KL-IG²':
                attr_cache[(m,cls)] = _attr_for_class_fast(m, x1, cls, klig2_fixed=klig2_fixed, path_fixed=path_fixed)
            else:
                attr_cache[(m,cls)] = _attr_for_class_fast(m, x1, cls)

    for ci, cj in itertools.combinations(high_cls, 2):
        td = wup_tree_dist(ci, cj)
        if td is None: continue
        for m in METHODS:
            scatter_all[m].append((td, cosine_dist_cs(attr_cache[(m,ci)], attr_cache[(m,cj)])))
    pickle.dump(scatter_all, open(_cache_scatter, 'wb'))

print(f'\nScatter ready — {len(scatter_all[METHODS[0]])} points/method')

# ── Diagnostic: signed rho, tree-distance vs attribution-distance ────────────
print("\n=== Wu-Palmer tree-distance rho (want positive) ===")
from scipy.stats import spearmanr as _sr
for m in METHODS:
    pts = scatter_all[m]
    td  = np.array([p[0] for p in pts]); ad = np.array([p[1] for p in pts])
    rho, pval = _sr(td, ad)
    print(f"  {m:25s}  n={len(pts):5d}  rho={rho:+.3f}  p={pval:.2g}")


multi_imgs_scatter: 104 images, 4 tree bins
  [0.00,0.25): 30
  [0.25,0.50): 30
  [0.50,0.75): 30
  [0.75,1.01): 14
Scatter needs CF images for 80 distinct y2 classes


scatter CF pool:  98%|█████████▊| 57/58 [02:31<00:02,  2.65s/it]


scatter-CF pool: 80/80 classes


class-sens scatter (Wu-Palmer): 100%|██████████| 104/104 [09:08<00:00,  5.28s/it]


Scatter ready — 132 points/method

=== Wu-Palmer tree-distance rho (want positive) ===
  KLIG-Adaptive              n=  132  rho=+0.114  p=0.19
  KL-IG (linear)             n=  132  rho=+0.044  p=0.62
  KL-IG²                     n=  132  rho=+0.196  p=0.024
  KL-IG² (adaptive)          n=  132  rho=-0.024  p=0.79


In [48]:
# ── Class Sensitivity Scatter — Step 3: plot ──────────────────────────────────
from scipy.stats import spearmanr

N_BINS = 10   # bins for error-bar summary (CLIP dist is continuous)

fig, axes = plt.subplots(1, len(METHODS), figsize=(7 * len(METHODS), 6),
                          facecolor='white', squeeze=False)
axes = axes[0]

for ax, m in zip(axes, METHODS):
    pts = scatter_all[m]
    if not pts:
        ax.set_title(f'{m}\nn/a', fontsize=10)
        continue
    dists = np.array([p[0] for p in pts])
    divs  = 1 - np.array([p[1] for p in pts])   # cosine similarity (1 - cos dist)

    # Scatter — colour by CLIP distance
    ax.scatter(dists, divs, alpha=0.30, s=15,
               c=dists, cmap='plasma', edgecolors='none')

    # Binned mean ± std error bars
    bin_edges = np.linspace(dists.min(), dists.max(), N_BINS + 1)
    bin_cx, bin_mu, bin_sd = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (dists >= lo) & (dists < hi)
        if mask.sum() >= 2:
            bin_cx.append((lo + hi) / 2)
            bin_mu.append(divs[mask].mean())
            bin_sd.append(divs[mask].std())
    if bin_cx:
        ax.errorbar(bin_cx, bin_mu, yerr=bin_sd,
                    fmt='o', color='black', ms=5, lw=1.4, zorder=5, capsize=3)

    # Regression line
    z  = np.polyfit(dists, divs, 1)
    xs = np.linspace(dists.min(), dists.max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), color=COLORS[m], lw=2, ls='--', alpha=0.8)

    rho_val, pval = spearmanr(dists, divs)
    # Report |ρ| (class-sensitivity magnitude), matching evaluation notebook
    ax.set_title(
        f'{m}\n|ρ| = {abs(rho_val):.3f}  (p = {pval:.3f})\nn = {len(pts)} pairs',
        fontsize=11, fontweight='bold', color=COLORS[m])
    ax.set_xlabel('CLIP semantic distance', fontsize=10)
    ax.set_ylabel('Attribution cosine similarity', fontsize=10)
    ax.set_xlim(dists.min() - 0.02, dists.max() + 0.02)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0)
    ax.set_axisbelow(True)
plt.suptitle(
    'Class Sensitivity: Attribution Cosine Similarity vs. CLIP Semantic Distance\n'
    'Similarity falls as classes diverge  |  higher |ρ| = more class-sensitive',
    fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


### Note on the CS ρ sign convention
The y-axis is **cosine similarity** of the two attribution maps (target class vs. competitor class).

- As CLIP distance between the two classes grows, a *class-sensitive* method makes the two maps **less similar**, so the raw Spearman correlation is **negative**.
- We therefore report **|ρ|** (the magnitude), exactly as the evaluation notebook does (`|Spearman ρ|`). **Higher |ρ| = more class-sensitive.**
- |ρ| ≈ 0 = class-agnostic: the map looks the same no matter which class is targeted.

This matches `evaluation_main_notebook_updated.ipynb`, where the raw ρ is also negative and the published ranking uses `abs(ρ)`.

**Why KLIG-Adaptive and KL-IG² (adaptive) can differ:**  
`cosine_dist_cs` clips attributions to [0, ∞) before comparing. KLIG-Adaptive produces bipolar maps; after clipping, both class maps keep a similar positive-region skeleton → higher similarity → smaller |ρ|. KL-IG²'s distribution-space path produces more class-discriminative positive regions → larger |ρ|.


In [ ]:
# ── Class Sensitivity Examples: KLIG-Adaptive vs KL-IG² (adaptive) ──────────
# Shows attribution maps for only Top-1 and Top-2 predicted classes.
# Methods shown:
#   1. KLIG-Adaptive
#   2. KL-IG² (adaptive)
#
# IMPORTANT:
# vmax is computed separately for EACH attribution map so one method does not
# wash out or over-saturate the other.

N_IMGS_VIZ_CS = 3
TOP_K_CLASSES = 2
MIN_CLIP_DIST = 0.10

VIZ_METHODS = ['KLIG-Adaptive', 'KL-IG² (adaptive)']
VIZ_COLORS  = {m: COLORS[m] for m in VIZ_METHODS}

IMAGE_TYPE_KEYWORDS = {
    "animal":  ["dog","cat","bird","fish","tench","goldfish","shark","stingray",
                "eel","trout","salmon","carp","pike","perch","bass","snake",
                "lizard","frog","elephant","tiger","lion","bear","fox","wolf",
                "deer","horse","cow","sheep","monkey","ape","panda","leopard",
                "zebra","penguin","owl","eagle","whale","spider","beetle",
                "rabbit","squirrel","goat","pig","camel","kangaroo","ostrich"],
    "vehicle": ["car","truck","bus","train","boat","ship","airplane","bicycle",
                "motorcycle","cab","wagon","trailer","scooter","ambulance"],
    "food":    ["bread","pizza","cake","apple","banana","orange","broccoli",
                "sandwich","hotdog","donut","cheese","mushroom","strawberry",
                "lemon","ice cream"],
}

NON_ANIMAL_CLS = set(range(400, 1000))   # ImageNet 0-399 ≈ animals
CS_PROB_THRESH = 0.10

def _image_type(cls_idx: int) -> str | None:
    label = imagenet_labels[cls_idx].lower()
    for type_name, keywords in IMAGE_TYPE_KEYWORDS.items():
        if any(kw in label for kw in keywords):
            return type_name
    return None


# ── Build candidate pool — non-animal classes only ────────────────────────────
import random
from itertools import zip_longest

def _make_candidate(x_tensor, idx):
    x1 = x_tensor.squeeze(0).to(DEVICE)
    with torch.no_grad():
        probs = model(x1.unsqueeze(0)).softmax(-1)[0].cpu()
    high = sorted(
        [int(c) for c in (probs > CS_PROB_THRESH).nonzero(as_tuple=True)[0]],
        key=lambda c: probs[c].item(), reverse=True,
    )
    if len(high) < 2 or high[0] not in NON_ANIMAL_CLS:
        return None
    cd = clip_semantic_dist(high[0], high[1])
    if cd < MIN_CLIP_DIST:
        return None
    return (cd, {
        'x': x_tensor, 'idx': idx,
        'high_cls':   high,
        'high_probs': [float(probs[c]) for c in high],
        'sig_adapt':  SIGMA_FINAL,
    })

# Try existing dataset first
candidates = []
for d in dataset:
    r = _make_candidate(d['x'], d['idx'])
    if r:
        candidates.append(r)

# Fall back to a fresh shuffled stream scan if not enough
if len(candidates) < N_IMGS_VIZ_CS:
    print(f"Only {len(candidates)} non-animal candidates in dataset — scanning stream...")
    from datasets import load_dataset as _hf
    _vs = _hf('evanarlian/imagenet_1k_resized_256', split='val', streaming=True)
    _vs = _vs.shuffle(seed=99, buffer_size=3000)
    _extra_idx = 20000
    for item in tqdm(_vs.take(3000), desc='stream scan'):
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        r = _make_candidate(preprocess(img).unsqueeze(0), _extra_idx)
        if r:
            candidates.append(r)
        _extra_idx += 1
        if len(candidates) >= N_IMGS_VIZ_CS * 10:
            break

print(f"Non-animal candidates: {len(candidates)}")
if not candidates:
    raise ValueError("No non-animal candidates found. Try lowering MIN_CLIP_DIST.")

random.shuffle(candidates)   # no fixed seed → different each run

# Bucket by keyword type
type_buckets: dict[str, list] = {t: [] for t in IMAGE_TYPE_KEYWORDS}
uncategorised: list = []
for cd, d in candidates:
    t = _image_type(d['high_cls'][0])
    (type_buckets[t] if t is not None else uncategorised).append((cd, d))

for t, b in type_buckets.items():
    print(f"  {t:8s}: {len(b)}")
print(f"  {'other':8s}: {len(uncategorised)}")

# Round-robin across categories so no single type dominates
# Round-robin across categories so no single type dominates
_non_empty     = [b for b in type_buckets.values() if b]
_interleaved   = [item for group in zip_longest(*_non_empty)
                  for item in group if item is not None]

selected     = []
used_idxs: set = set()
used_cls:  set = set()                     # ← dedup by top-1 class too

def _try_add(cd, d):
    c0 = d['high_cls'][0]
    if d['idx'] in used_idxs or c0 in used_cls:
        return
    selected.append((cd, d))
    used_idxs.add(d['idx'])
    used_cls.add(c0)

for cd, d in _interleaved:
    _try_add(cd, d)
    if len(selected) == N_IMGS_VIZ_CS:
        break

# Pad from uncategorised only if we still need more
for cd, d in uncategorised:
    if len(selected) == N_IMGS_VIZ_CS:
        break
    _try_add(cd, d)

random.shuffle(selected)

# ── Compute attribution maps ──────────────────────────────────────────────────
print("\nComputing attribution maps...")

viz_data = []

for clip_dist_val, d in tqdm(selected, desc='viz attrs'):
    x1 = d['x'].squeeze(0).to(DEVICE)

    high_cls   = d['high_cls'][:TOP_K_CLASSES]
    high_probs = d['high_probs'][:TOP_K_CLASSES]

    sig_adapt = d.get('sig_adapt', SIGMA_FINAL)
    H, W = x1.shape[1], x1.shape[2]

    x_cf_img = _pick_cf_scatter(d)
    if x_cf_img.dim() == 4:
        x_cf_img = x_cf_img.squeeze(0)
    x_cf_img = x_cf_img.to(DEVICE)

    klig2_adapt = None
    path_adapt  = None
    if 'KL-IG² (adaptive)' in METHODS:
        lv_fl = 2 * math.log(sig_adapt)
        klig2_adapt = _build_klig2(x_cf_img, sig_adapt, lv_fl)
        path_adapt  = _build_gradpath_once(klig2_adapt, x1)

    maps = {}
    for method in VIZ_METHODS:
        for cls in high_cls:
            maps[(method, cls)] = _attr_for_class_fast(
                method, x1, cls,
                klig2_fixed=None, klig2_adapt=klig2_adapt,
                path_fixed=None,  path_adapt=path_adapt,
                sig_adapt=sig_adapt,
            ).reshape(H, W)

    viz_data.append({
        'x': d['x'],
        'high_cls': high_cls,
        'high_probs': high_probs,
        'maps': maps,
        'clip_top12': clip_dist_val,
    })

print(f"Showing Top-{TOP_K_CLASSES} predicted classes per image")


# ── Plot ──────────────────────────────────────────────────────────────────────
N_ROWS = len(viz_data)
N_METH = len(VIZ_METHODS)
N_COLS = 1 + N_METH * TOP_K_CLASSES

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(2.6 * N_COLS, 2.8 * N_ROWS),
    facecolor='white'
)
if N_ROWS == 1:
    axes = axes[np.newaxis, :]


def _add_banner(text, col_start, col_end, color):
    ax_l = axes[0, col_start]
    ax_r = axes[0, col_end]
    bb_l = ax_l.get_position()
    bb_r = ax_r.get_position()
    fig.text(
        (bb_l.x0 + bb_r.x1) / 2,
        bb_l.y1 + 0.025,
        text,
        fontsize=12, fontweight='bold',
        ha='center', va='bottom', color=color
    )


for row, item in enumerate(viz_data):
    img = np.clip(
        denormalize(item['x'][0]).permute(1, 2, 0).cpu().numpy(), 0, 1)

    high_cls   = item['high_cls']
    high_probs = item['high_probs']

    ax0 = axes[row, 0]
    ax0.imshow(img)
    ax0.set_xticks([]); ax0.set_yticks([])
    for sp in ax0.spines.values():
        sp.set_visible(False)
    ax0.text(
        0.5, -0.08,
        f"CLIP(T1,T2): {item['clip_top12']:.3f}",
        transform=ax0.transAxes,
        fontsize=9, fontweight='bold', va='top', ha='center'
    )

    for m_i, method in enumerate(VIZ_METHODS):
        for c_i, cls in enumerate(high_cls):
            col = 1 + m_i * TOP_K_CLASSES + c_i
            ax  = axes[row, col]

            a = item['maps'][(method, cls)]
            if torch.is_tensor(a):
                a = a.detach().cpu().numpy()

            vmax = max(np.percentile(np.abs(a), 99), 1e-9)
            ax.imshow(a, cmap='RdBu_r', vmin=-vmax, vmax=vmax)

            short_label = imagenet_labels[cls].split(',')[0][:16]
            ax.set_title(
                f"Top-{c_i + 1}: {short_label}\np={high_probs[c_i]:.2f}",
                fontsize=8
            )
            ax.axis('off')

axes[0, 0].set_title('Original', fontsize=11, fontweight='bold')

for m_i, method in enumerate(VIZ_METHODS):
    col_start = 1 + m_i * TOP_K_CLASSES
    col_end   = col_start + TOP_K_CLASSES - 1
    _add_banner(method, col_start, col_end, VIZ_COLORS[method])

for m_i in range(1, N_METH):
    left_ax  = axes[0, m_i * TOP_K_CLASSES]
    right_ax = axes[0, m_i * TOP_K_CLASSES + 1]
    bb_l = left_ax.get_position()
    bb_r = right_ax.get_position()
    sep_x = (bb_l.x1 + bb_r.x0) / 2
    fig.add_artist(plt.Line2D(
        [sep_x, sep_x], [0.02, 0.96],
        color='#888', lw=1.5, transform=fig.transFigure
    ))

plt.suptitle(
    'Class Sensitivity Examples: KLIG-Adaptive vs KL-IG² Adaptive\n'
    '(Top-1 and Top-2 predicted classes only; per-map vmax)',
    fontsize=13, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.subplots_adjust(hspace=0.55)
plt.savefig(
    'klig2_cs_viz_top2_adaptive_only_permap_vmax.png',
    dpi=180, bbox_inches='tight'
)
plt.show()


# ── Print selected class summary ──────────────────────────────────────────────
print(f"\n{'CLIP(T1-T2)':>12}  Top-2 Classes (prob)")
print('-' * 80)
for item in viz_data:
    cls_str = ', '.join(
        f"{imagenet_labels[c].split(',')[0][:10]}({p:.2f})"
        for c, p in zip(item['high_cls'], item['high_probs'])
    )
    print(f"{item['clip_top12']:>12.3f}  {cls_str}")

In [ ]:
# ── KL-IG vs KL-IG²: single-image class-sensitivity contrast ─────────────────
# Honest framing — this is a METHOD comparison, NOT a baseline-only ablation:
#   Row 0  KLIG-Adaptive       — linear path from the N(0,1) prior (no counterfactual)
#   Row 1  KL-IG² (adaptive)   — rep-descent path anchored to the Top-2 counterfactual
# The two methods differ in path + baseline + (attr vs attr_mu); we do NOT claim
# "only the baseline differs". We pick the image where the class-sensitivity gap
# (d_attr) between the two methods is largest, and report d_attr straight from the
# maps shown. This is the same method pair behind the aggregate CS-ρ diagnostic
# (KLIG-Adaptive |ρ|≈0.008 vs KL-IG² (adaptive) |ρ|≈0.23), so the numbers are
# consistent with the scatter.
import numpy as _np
import matplotlib.pyplot as plt

D_SEM_MIN = 0.40                    # skip near-synonym class pairs (pseudocode threshold)
M_KLIG    = 'KLIG-Adaptive'
M_KLIG2   = 'KL-IG² (adaptive)'


def _four_maps(d):
    """(x1, c1, c2, maps) with maps[(method, cls)] an (H,W) np array.
    Mirrors the cell-26 scatter computation exactly: per-class adaptive sigma,
    per-class rep-descent path for KL-IG² (adaptive); per-class sigma for KLIG-Adaptive."""
    x1 = d['x'].squeeze(0).to(DEVICE)
    H, W = x1.shape[1], x1.shape[2]
    c1, c2 = d['high_cls'][0], d['high_cls'][1]

    x_cf = _pick_cf_scatter(d)
    if x_cf.dim() == 4:
        x_cf = x_cf.squeeze(0)
    x_cf = x_cf.to(DEVICE)

    sig_cache = dict(d.get('sig_per_cls', {}))
    def _sig(cls):
        if cls not in sig_cache:
            sig_cache[cls] = find_sigma_stop(model, x1, int(cls),
                                             tau=0.95, n_samples=32, n_iter=12)
        return sig_cache[cls]

    maps = {}
    for cls in (c1, c2):
        sc = _sig(cls)
        # KL-IG² (adaptive): per-class rep-descent path anchored to x_cf
        k2  = _build_klig2(x_cf, sc, 2 * math.log(sc))
        pth = _build_gradpath_once(k2, x1)
        maps[(M_KLIG2, cls)] = _attr_for_class_fast(
            M_KLIG2, x1, cls, klig2_adapt=k2, path_adapt=pth, sig_adapt=sc
        ).reshape(H, W)
        # KLIG-Adaptive: linear path from the prior (no counterfactual)
        maps[(M_KLIG, cls)] = _attr_for_class_fast(
            M_KLIG, x1, cls, sig_adapt=sc
        ).reshape(H, W)
    return x1, c1, c2, maps


# ── 1. find the image with the largest class-sensitivity gap ─────────────────
_cands = [d for d in multi_imgs_scatter if len(d.get('high_cls', [])) >= 2]
_qual  = [d for d in _cands
          if clip_semantic_dist(d['high_cls'][0], d['high_cls'][1]) >= D_SEM_MIN]
if _qual:
    _pool = _qual
    print(f"{len(_pool)}/{len(_cands)} candidate images pass d_sem >= {D_SEM_MIN}")
else:
    _pool = _cands
    print(f"No image passed d_sem >= {D_SEM_MIN}; using all {len(_cands)} candidates")

best = None
for d in tqdm(_pool, desc='KL-IG vs KL-IG² contrast'):
    x1, c1, c2, maps = _four_maps(d)
    d_klig  = cosine_dist_cs(maps[(M_KLIG,  c1)], maps[(M_KLIG,  c2)])   # want LOW  (class-blind)
    d_klig2 = cosine_dist_cs(maps[(M_KLIG2, c1)], maps[(M_KLIG2, c2)])   # want HIGH (class-sensitive)
    contrast = d_klig2 - d_klig
    if best is None or contrast > best['contrast']:
        best = dict(x=d['x'], c1=c1, c2=c2, maps=maps,
                    d_klig=d_klig, d_klig2=d_klig2, contrast=contrast,
                    d_sem=clip_semantic_dist(c1, c2))

c1, c2 = best['c1'], best['c2']
lbl1 = imagenet_labels[c1].split(',')[0]
lbl2 = imagenet_labels[c2].split(',')[0]
print(f"\nChosen image: Top-1 {lbl1!r} / Top-2 {lbl2!r}   d_sem={best['d_sem']:.2f}")
print(f"  d_attr  {M_KLIG:<18s} = {best['d_klig']:.3f}   (lower  = class-blind)")
print(f"  d_attr  {M_KLIG2:<18s} = {best['d_klig2']:.3f}   (higher = class-sensitive)")
print(f"  contrast (KL-IG² - KL-IG)        = {best['contrast']:.3f}")


# ── 2. draw: 2 rows (method) x 3 cols (original | why c1 | why c2) ───────────
ROWS = [(M_KLIG,  best['d_klig'],  COLORS[M_KLIG]),
        (M_KLIG2, best['d_klig2'], COLORS[M_KLIG2])]
_more = M_KLIG2 if best['d_klig2'] >= best['d_klig'] else M_KLIG   # which row gets the ✓

img = _np.clip(denormalize(best['x'][0]).permute(1, 2, 0).cpu().numpy(), 0, 1)
fig, axes = plt.subplots(2, 3, figsize=(9.5, 6.6), facecolor='white')

for r, (method, d_attr, color) in enumerate(ROWS):
    # shared per-row (per-method) vmax across this method's two maps → maps comparable
    both = _np.concatenate([_np.abs(best['maps'][(method, c1)]).ravel(),
                            _np.abs(best['maps'][(method, c2)]).ravel()])
    vmax = max(float(_np.percentile(both, 99)), 1e-9)

    axes[r, 0].imshow(img)
    axes[r, 0].set_title('Original' if r == 0 else 'same image',
                         fontsize=10, fontweight='bold')
    for cc, cls in enumerate((c1, c2), start=1):
        ax = axes[r, cc]
        ax.imshow(best['maps'][(method, cls)], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f"Why {imagenet_labels[cls].split(',')[0][:16]}?", fontsize=10)
    for cc in range(3):
        axes[r, cc].set_xticks([]); axes[r, cc].set_yticks([])

    verdict = 'maps differ ✓' if method == _more else 'maps alike'
    axes[r, 0].text(-0.18, 0.5, f"{method}\n$d_{{attr}}$ = {d_attr:.2f}\n{verdict}",
                    transform=axes[r, 0].transAxes, rotation=90,
                    va='center', ha='center', fontsize=11, fontweight='bold', color=color)

fig.suptitle(
    f"Class sensitivity on one image — KL-IG vs KL-IG²   "
    f"(Top-1 {lbl1} / Top-2 {lbl2},  d_sem={best['d_sem']:.2f})\n"
    f"KL-IG² (adaptive) anchors a rep-descent path to the Top-2 counterfactual; "
    f"KLIG-Adaptive integrates a linear path from the prior.\n"
    f"Different methods (path + baseline) — d_attr read directly from the maps shown.",
    fontsize=10.5, fontweight='bold')
plt.tight_layout(rect=[0.04, 0, 1, 0.92])
plt.savefig('klig_vs_klig2.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
# ── Counterfactual Ablation Study ─────────────────────────────────────────────
# Three CF strategies for KL-IG²:
#   'y2'       : second-most-likely class (current default)
#   'random'   : random class from cf_pool
#   'furthest' : lowest-probability class in model output

import random as _rnd
_rnd.seed(0)

CF_STRATEGIES   = ['y2', 'random', 'furthest']
CF_COLORS       = {'y2': '#e41a1c', 'random': '#ff7f00', 'furthest': '#4dac26'}
CF_LABELS       = {'y2': 'KL-IG² (y₂)', 'random': 'KL-IG² (random CF)',
                   'furthest': 'KL-IG² (furthest CF)'}

_cache_cf_abl   = CACHE_DIR / 'klig2_cf_ablation_scatter.pkl'
_cache_cf_abl_n = CACHE_DIR / 'klig2_cf_ablation_scatter_n.pkl'

# ── CF picker for each strategy ───────────────────────────────────────────────
def _pick_cf_strategy(d, strategy):
    """Return CF image (C,H,W) on CPU for the given strategy."""
    with torch.no_grad():
        logits = model(d['x'].to(DEVICE))[0]
        probs  = logits.softmax(0)

    if strategy == 'y2':
        cls = d['high_cls'][1] if len(d['high_cls']) > 1 else d['high_cls'][0]

    elif strategy == 'random':
        available = list(cf_pool.keys())
        # exclude top-1 to keep it meaningful
        available = [c for c in available if c != d['high_cls'][0]] or available
        cls = _rnd.choice(available)

    elif strategy == 'furthest':
        # lowest softmax probability among classes we actually have a CF image for
        pool_classes = torch.tensor(list(cf_pool.keys()))
        cls = int(pool_classes[probs[pool_classes].argmin()].item())

    if cls in cf_pool:
        img = cf_pool[cls]
    else:
        img = next(iter(cf_pool.values()))
    return img.squeeze(0).to(DEVICE) if img.dim() == 4 else img.to(DEVICE), cls


# ── Scatter computation ───────────────────────────────────────────────────────
if not FORCE_RECOMPUTE and _cache_cf_abl.exists():
    abl_scatter = pickle.load(open(_cache_cf_abl, 'rb'))
    abl_done_n  = pickle.load(open(_cache_cf_abl_n, 'rb'))
    print(f'[resume] {sum(len(v) for v in abl_scatter.values())} points, '
          f'{abl_done_n}/{len(multi_imgs_scatter)} done')
else:
    abl_scatter = {s: [] for s in CF_STRATEGIES}
    abl_done_n  = 0

for d in tqdm(multi_imgs_scatter[abl_done_n:], desc='CF ablation scatter'):
    x1       = d['x'].squeeze(0).to(DEVICE)
    high_cls = d['high_cls']

    attr_cache = {}
    for strategy in CF_STRATEGIES:
        x_cf_img, _cf_cls = _pick_cf_strategy(d, strategy)
        klig2 = _build_klig2(x_cf_img, SIGMA_FINAL, LV_FLOOR)
        path  = _build_gradpath_once(klig2, x1)

        for cls in high_cls:
            key = (strategy, cls)
            if key not in attr_cache:
                attr_cache[key] = _attr_for_class_fast(
                    'KL-IG²', x1, cls, klig2_fixed=klig2, path_fixed=path)

    for ci, cj in itertools.combinations(high_cls, 2):
        cl_dist = clip_semantic_dist(ci, cj)
        for strategy in CF_STRATEGIES:
            cd = cosine_dist_cs(attr_cache[(strategy, ci)],
                                attr_cache[(strategy, cj)])
            abl_scatter[strategy].append((cl_dist, cd))

    abl_done_n += 1
    pickle.dump(abl_scatter, open(_cache_cf_abl,   'wb'))
    pickle.dump(abl_done_n,  open(_cache_cf_abl_n, 'wb'))

print(f'Done — {len(abl_scatter["y2"])} points per strategy')

In [ ]:
# ── CF Ablation: plots + table ────────────────────────────────────────────────
from scipy.stats import spearmanr

# ── 1. Attribution map examples (3 images, 3 strategies side by side) ────────
N_VIZ_ABL = 3
_abl_imgs  = [d for d in multi_imgs_scatter
              if (d.get('clip_dist') or
                  clip_semantic_dist(d['high_cls'][0], d['high_cls'][1])) > 0.35
              ][:N_VIZ_ABL]

N_COLS_ABL = 1 + len(CF_STRATEGIES)   # orig + 3 strategies
fig_v, axes_v = plt.subplots(
    N_VIZ_ABL, N_COLS_ABL,
    figsize=(2.8 * N_COLS_ABL, 3.0 * N_VIZ_ABL),
    facecolor='white',
    gridspec_kw={'hspace': 0.4, 'wspace': 0.05},
)
if N_VIZ_ABL == 1: axes_v = axes_v[np.newaxis, :]

for r, d in enumerate(tqdm(_abl_imgs, desc='viz ablation')):
    x1  = d['x'].squeeze(0).to(DEVICE)
    tgt = d['high_cls'][0]

    # col 0: original
    ax = axes_v[r, 0]
    ax.imshow(np.clip(denormalize(x1.cpu()).permute(1,2,0).numpy(), 0, 1))
    ax.set_title(f"{imagenet_labels[tgt].split(',')[0][:16]}\np={d['high_probs'][0]:.2f}",
                 fontsize=9, fontweight='bold')
    ax.axis('off')

    for ci, strategy in enumerate(CF_STRATEGIES):
        x_cf_img, cf_cls = _pick_cf_strategy(d, strategy)
        cf_lbl = imagenet_labels[cf_cls].split(',')[0][:14]

        klig2 = _build_klig2(x_cf_img, SIGMA_FINAL, LV_FLOOR)
        path  = _build_gradpath_once(klig2, x1)
        a = _attr_for_class_fast('KL-IG²', x1, tgt,
                                  klig2_fixed=klig2, path_fixed=path)
        a = a.reshape(x1.shape[1], x1.shape[2])
        vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)

        ax = axes_v[r, 1 + ci]
        ax.imshow(a, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f"{CF_LABELS[strategy]}\nCF: {cf_lbl}",
                     fontsize=8, color=CF_COLORS[strategy], fontweight='bold')
        ax.axis('off')

plt.suptitle('KL-IG² CF Ablation — Attribution Maps\n'
             'Same image, same target class, different counterfactual',
             fontsize=12, fontweight='bold', y=1.02)
plt.savefig('klig2_cf_ablation_maps.png', dpi=180, bbox_inches='tight')
plt.show()

# ── 2. Scatter plot: one panel per strategy ───────────────────────────────────
fig_s, axes_s = plt.subplots(
    1, len(CF_STRATEGIES),
    figsize=(7 * len(CF_STRATEGIES), 5),
    facecolor='white',
)

for ax, strategy in zip(axes_s, CF_STRATEGIES):
    pts   = abl_scatter[strategy]
    dists = np.array([p[0] for p in pts])
    divs  = np.array([p[1] for p in pts])
    rho, pval = spearmanr(dists, divs)

    ax.scatter(dists, divs, alpha=0.3, s=15,
               c=dists, cmap='plasma', edgecolors='none')
    z  = np.polyfit(dists, divs, 1)
    xs = np.linspace(dists.min(), dists.max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), color=CF_COLORS[strategy], lw=2, ls='--')

    ax.set_title(f"{CF_LABELS[strategy]}\nρ={rho:.3f}  p={pval:.4f}  n={len(pts)}",
                 fontsize=11, fontweight='bold', color=CF_COLORS[strategy])
    ax.set_xlabel('CLIP semantic distance', fontsize=10)
    ax.set_ylabel('Cosine dist (attr_c₁ vs attr_c₂)', fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)

plt.suptitle('CF Ablation: Class Sensitivity Scatter per CF Strategy',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('klig2_cf_ablation_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. Summary table ──────────────────────────────────────────────────────────
print('\nCF Ablation — Class Sensitivity Summary')
print(f"{'Strategy':<25}  {'CS ρ':>7}  {'p-value':>9}  {'n':>6}  {'Interpretation'}")
print('─' * 72)
interp = {
    'y2':       'decision boundary CF  → highest CS (design intent)',
    'random':   'unrelated CF         → moderate/low CS',
    'furthest': 'maximally distant CF → should degrade CS',
}
for strategy in CF_STRATEGIES:
    pts  = abl_scatter[strategy]
    dists = np.array([p[0] for p in pts])
    divs  = np.array([p[1] for p in pts])
    rho, pval = spearmanr(dists, divs)
    print(f"{CF_LABELS[strategy]:<25}  {rho:>7.3f}  {pval:>9.4f}  "
          f"{len(pts):>6}  {interp[strategy]}")

In [ ]:
# ── Class Sensitivity Diagnostic — Feature Agreement (Williamson et al. 2025) ─
# Protocol from CASE paper (arxiv:2506.07327):
#   For each image, generate attribution maps for top-1 and top-2 class labels.
#   Compute top-5% feature agreement: F(E,E',k) = |top-k(E) ∩ top-k(E')| / k
#   One-sided Wilcoxon test: H0: median(F) >= 0.50
#   Reject H0 (p < 0.05) → method produces class-distinct explanations.

from scipy.stats import wilcoxon

TOP_K_PCT      = 0.05    # top 5% of pixels, same as CASE paper
FA_VIZ_METHODS = METHODS  # run for ALL methods defined in notebook

def feature_agreement(e1: np.ndarray, e2: np.ndarray, k_pct: float = TOP_K_PCT) -> float:
    """Proportion of top-k pixels shared between two attribution maps."""
    e1 = np.abs(e1.ravel()).astype(np.float64)
    e2 = np.abs(e2.ravel()).astype(np.float64)
    k  = max(1, int(len(e1) * k_pct))
    top1 = set(np.argpartition(e1, -k)[-k:].tolist())
    top2 = set(np.argpartition(e2, -k)[-k:].tolist())
    return len(top1 & top2) / k


# ── Collect agreement scores ──────────────────────────────────────────────────
fa_scores = {m: [] for m in FA_VIZ_METHODS}

for d in tqdm(multi_imgs_scatter, desc='feature agreement'):
    x1       = d['x'].squeeze(0).to(DEVICE)
    high_cls = d['high_cls'][:2]
    if len(high_cls) < 2:
        continue
    sig_adapt = d.get('sig_adapt', SIGMA_FINAL)
    H, W      = x1.shape[1], x1.shape[2]

    x_cf_img = _pick_cf_scatter(d)
    if x_cf_img.dim() == 4:
        x_cf_img = x_cf_img.squeeze(0)
    x_cf_img = x_cf_img.to(DEVICE)

    klig2_fixed = path_fixed = None
    if 'KL-IG²' in FA_VIZ_METHODS:
        klig2_fixed = _build_klig2(x_cf_img, SIGMA_FINAL, LV_FLOOR)
        path_fixed  = _build_gradpath_once(klig2_fixed, x1)

    klig2_adapt = path_adapt = None
    if 'KL-IG² (adaptive)' in FA_VIZ_METHODS:
        lv_fl       = 2 * math.log(sig_adapt)
        klig2_adapt = _build_klig2(x_cf_img, sig_adapt, lv_fl)
        path_adapt  = _build_gradpath_once(klig2_adapt, x1)

    for method in FA_VIZ_METHODS:
        e1 = _attr_for_class_fast(
            method, x1, int(high_cls[0]),
            klig2_fixed=klig2_fixed, klig2_adapt=klig2_adapt,
            path_fixed=path_fixed,   path_adapt=path_adapt,
            sig_adapt=sig_adapt,
        ).reshape(H, W)
        e2 = _attr_for_class_fast(
            method, x1, int(high_cls[1]),
            klig2_fixed=klig2_fixed, klig2_adapt=klig2_adapt,
            path_fixed=path_fixed,   path_adapt=path_adapt,
            sig_adapt=sig_adapt,
        ).reshape(H, W)
        fa_scores[method].append(feature_agreement(e1, e2))


# ── Statistical test ──────────────────────────────────────────────────────────
print(f"\nClass Sensitivity — Top-{TOP_K_PCT*100:.0f}% Feature Agreement")
print(f"H0: median(F) >= 0.50  (one-sided Wilcoxon signed-rank, α=0.05)\n")
print(f"{'Method':25s}  {'n':>5}  {'mean':>6}  {'median':>7}  {'std':>6}  {'p-value':>10}  {'H0 rejected?':>13}")
print('─' * 82)

results = {}
for method in FA_VIZ_METHODS:
    scores  = np.array(fa_scores[method])
    stat, p = wilcoxon(scores - 0.50, alternative='less')
    reject  = 'YES ✓' if p < 0.05 else 'NO  ✗'
    results[method] = {'scores': scores, 'p': p, 'reject': p < 0.05}
    print(f"{method:25s}  {len(scores):>5}  {scores.mean():>6.3f}  "
          f"{np.median(scores):>7.3f}  {scores.std():>6.3f}  {p:>10.4f}  {reject:>13}")


# ── Distribution plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='white')

# Left: violin
ax = axes[0]
data  = [results[m]['scores'] for m in FA_VIZ_METHODS]
parts = ax.violinplot(data, positions=range(len(FA_VIZ_METHODS)),
                      showmedians=True, showextrema=False)
for pc, method in zip(parts['bodies'], FA_VIZ_METHODS):
    pc.set_facecolor(COLORS[method])
    pc.set_alpha(0.55)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(2)
ax.axhline(0.50, color='red', linestyle='--', lw=1.5, label='H0 threshold (0.50)')
ax.set_xticks(range(len(FA_VIZ_METHODS)))
ax.set_xticklabels(FA_VIZ_METHODS, fontsize=9, rotation=15, ha='right')
ax.set_ylabel(f'Top-{TOP_K_PCT*100:.0f}% Feature Agreement', fontsize=10)
ax.set_title('Agreement Distribution\n(lower = more class-sensitive)', fontsize=10)
ax.legend(fontsize=9); ax.set_ylim(0, 1)

# Right: CDF
ax = axes[1]
for method in FA_VIZ_METHODS:
    scores = np.sort(results[method]['scores'])
    cdf    = np.arange(1, len(scores) + 1) / len(scores)
    p_val  = results[method]['p']
    reject = '✓' if results[method]['reject'] else '✗'
    label  = f"{method}  p={p_val:.4f} {reject}"
    ax.plot(scores, cdf, color=COLORS[method], lw=2, label=label)
ax.axvline(0.50, color='red', linestyle='--', lw=1.5, label='H0 threshold')
ax.set_xlabel(f'Top-{TOP_K_PCT*100:.0f}% Feature Agreement', fontsize=10)
ax.set_ylabel('CDF', fontsize=10)
ax.set_title('CDF of Agreement Scores\n(curve left of 0.5 = class-sensitive)', fontsize=10)
ax.legend(fontsize=8, loc='upper left'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'CASE Diagnostic: Top-{TOP_K_PCT*100:.0f}% Feature Agreement (Williamson et al. 2025)\n'
    'H₀: median agreement ≥ 0.50 — reject (p<0.05) → method is class-sensitive',
    fontsize=11, fontweight='bold', y=1.03,
)
plt.tight_layout()
plt.savefig('klig2_case_diagnostic.png', dpi=180, bbox_inches='tight')
plt.show()

# Attribution Maps & Path Build-up

In [ ]:
# ── Attribution maps ─────────────────────────────────────────────────────────
import random

N_VIS = 5
vis_idxs = random.sample(range(len(dataset)), min(N_VIS, len(dataset)))

fig, axes = plt.subplots(
    len(vis_idxs), 1 + len(METHODS),
    figsize=(2.6 * (1 + len(METHODS)), 2.6 * len(vis_idxs)),
    facecolor='white', squeeze=False,
)
for r, i in enumerate(vis_idxs):
    row_v   = dataset[i]
    img_np  = np.clip(denormalize(row_v['x'][0]).permute(1,2,0).numpy(), 0, 1)
    label   = imagenet_labels[row_v['target']].split(',')[0][:20]
    axes[r,0].imshow(img_np); axes[r,0].axis('off')
    # show label as the title of the original image (visible even with axis off)
    title = f'Original\n{label}' if r == 0 else label
    axes[r,0].set_title(title, fontsize=9, fontweight='bold')
    for c, m in enumerate(METHODS, start=1):
        a    = all_attrs[m][i].numpy()
        vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)
        axes[r,c].imshow(a, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[r,c].axis('off')
        if r == 0: axes[r,c].set_title(m, fontsize=9, fontweight='bold', color=COLORS[m])
plt.suptitle('Attribution maps', fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout(); plt.show()

In [ ]:
def _find_cf_for_class(cls, max_scan=4000, accept_prob=0.30):
    """Find an image that best represents `cls`. Prefers cf_pool, then the
    already-loaded dataset, then a stream scan — keeping the highest-prob
    candidate rather than requiring argmax==cls (handles rare classes)."""
    # 1) existing CF pool
    if 'cf_pool' in globals() and cls in cf_pool:
        return cf_pool[cls]

    best_x, best_p = None, -1.0

    # 2) check already-loaded dataset (cheap, no download)
    for d in dataset:
        with torch.no_grad():
            p = model(d['x'].to(DEVICE)).softmax(-1)[0, cls].item()
        if p > best_p:
            best_p, best_x = p, d['x'].to(DEVICE)
        if best_p >= accept_prob:
            return best_x

    # 3) stream scan, keep best-by-prob
    from datasets import load_dataset as _hf
    _s = _hf('evanarlian/imagenet_1k_resized_256', split='val',
             streaming=True).shuffle(seed=7, buffer_size=3000)
    for item in tqdm(_s.take(max_scan), desc=f'scan CF for {imagenet_labels[cls][:18]}'):
        im = item['image']
        if im.mode != 'RGB': im = im.convert('RGB')
        xx = preprocess(im).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            p = model(xx).softmax(-1)[0, cls].item()
        if p > best_p:
            best_p, best_x = p, xx
        if best_p >= accept_prob:
            break

    print(f'  best CF candidate for {imagenet_labels[cls]}: p={best_p:.3f}')
    return best_x   # never None as long as dataset is non-empty

In [ ]:
# ── KL-IG² Path Attribution: target | CF | attribution (CF→target) ────────────
# Auto-selects an image whose y2 (competitor) already has a real CF in cf_pool.
from scipy.ndimage import gaussian_filter
import random

PA_N_MC    = 24
PA_SMOOTH  = 0.8
PA_IDX     = None     # set an int to force a specific image; None = auto-pick

# ── Find dataset images whose y2 has a genuine CF in the pool ─────────────────
def _y2_of(i):
    with torch.no_grad():
        p = model(dataset[i]['x'].to(DEVICE)).softmax(-1)[0].cpu()
    o = p.argsort(descending=True).tolist()
    y2 = o[1] if o[0] == dataset[i]['target'] else o[0]
    return y2, float(p[dataset[i]['target']]), float(p[y2])

candidates = []
for i in range(len(dataset)):
    y2, p_t, p_y2 = _y2_of(i)
    if 'cf_pool' in globals() and y2 in cf_pool:
        candidates.append((i, y2, p_t, p_y2))

if not candidates:
    raise RuntimeError('No dataset image has its y2 in cf_pool — rebuild cf_pool first.')

if PA_IDX is None:
    # prefer a confident, well-separated pair; pick randomly among the top ones
    candidates.sort(key=lambda c: c[3], reverse=True)   # strongest competitor first
    PA_IDX, y2_pa, _, _ = random.choice(candidates[:max(5, len(candidates)//4)])
else:
    y2_pa = next(y2 for i, y2, _, _ in candidates if i == PA_IDX)

row_pa = dataset[PA_IDX]
x_pa   = row_pa['x'].to(DEVICE)
x1_pa  = x_pa.squeeze(0)
tgt_pa = row_pa['target']
with torch.no_grad():
    probs_pa = model(x_pa).softmax(-1)[0].cpu()

print(f'Picked idx={PA_IDX}')
print(f'Target : {imagenet_labels[tgt_pa]}  (p={probs_pa[tgt_pa]:.2f})')
print(f'CF (y2): {imagenet_labels[y2_pa]}  (p={probs_pa[y2_pa]:.2f})   [from cf_pool ✓]')

x_cf_1 = cf_pool[y2_pa].squeeze(0).to(DEVICE) if cf_pool[y2_pa].dim() == 4 \
         else cf_pool[y2_pa].to(DEVICE)

# ── Build KL-IG² GradPath, integrate target logit along it ────────────────────
sig_pa = find_sigma_stop(model, x_pa, tgt_pa, tau=0.95, n_samples=32, n_iter=12)
_ig2_pa = KLIGSquared(
    model, phi, x_cf_1,
    T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
    n_mc_path=N_MC_DESCENT, n_mc_grad=PA_N_MC,
    sigma_start=sig_pa, loss_stop=LOSS_STOP,
    lv_floor=2.0 * math.log(max(sig_pa, 1e-7)), lv_ceil=LV_CEIL,
    mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True, device=DEVICE,
)
_res_pa  = _ig2_pa.attribute(x1_pa, target=tgt_pa)
attr_pa  = absmax_collapse(_res_pa.attr_mu).cpu().numpy()
if PA_SMOOTH > 0:
    attr_pa = gaussian_filter(attr_pa, sigma=PA_SMOOTH)
K_pa = len(_res_pa.traj_mu) - 1
print(f'KL-IG² path: {K_pa} steps   |   sigma_start={sig_pa:.3f}')

# ── Plot: target | CF | attribution | overlay ─────────────────────────────────
def _rgb(t_chw):
    return denormalize(t_chw.cpu()).clamp(0, 1).permute(1, 2, 0).numpy()

img_tgt = _rgb(x1_pa)
img_cf  = _rgb(x_cf_1)
vmax    = max(float(np.percentile(np.abs(attr_pa), 99.5)), 1e-9)

fig, axes = plt.subplots(1, 4, figsize=(15, 4), facecolor='white')
axes[0].imshow(img_tgt)
axes[0].set_title(f'Target\n{imagenet_labels[tgt_pa].split(",")[0][:22]}',
                  fontsize=10, fontweight='bold')
axes[1].imshow(img_cf)
axes[1].set_title(f'Counterfactual = y2\n{imagenet_labels[y2_pa].split(",")[0][:22]}',
                  fontsize=10, fontweight='bold')
im = axes[2].imshow(attr_pa, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[2].set_title('KL-IG² attribution\n(CF → target)', fontsize=10, fontweight='bold')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
axes[3].imshow(img_tgt)
pos = np.clip(attr_pa, 0, None); pos = pos / (pos.max() + 1e-9)
axes[3].imshow(pos, cmap='hot', alpha=0.55)
axes[3].set_title('Overlay\n(positive evidence)', fontsize=10, fontweight='bold')
for ax in axes:
    ax.axis('off')
plt.suptitle('KL-IG²: attribution of the target image relative to the counterfactual baseline',
             fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('klig2_path_attr_cf_to_target.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# -- Path Attribution Build-up -- KLIG & KL-IG2 all variants ------------------
# Every method now reads as  baseline -> explicand:
#   KLIG methods (LinearPath) : zero baseline -> explicand
#   KL-IG2 methods (GradPath) : CF baseline   -> explicand  (reversed indexing)
# Layout: original | baseline | attr@25% | attr@50% | attr@75% | attr@100%

VIS_IMG_IDX      = 25
_BUILDUP_METHODS = ['KLIG-Adaptive', 'KL-IG (linear)', 'KL-IG²', 'KL-IG² (adaptive)']
T_STEPS          = [0.25, 0.5, 0.75, 1.0]
N_T              = len(T_STEPS)
N_MC_BUILDUP     = 10

# -- Pick one image -----------------------------------------------------------
BUILDUP_IDX = VIS_IMG_IDX

row_v   = dataset[BUILDUP_IDX]
x_viz   = row_v['x'].to(DEVICE)
x1_viz  = x_viz.squeeze(0)
tgt_viz = row_v['target']
meta_v  = all_path_meta[BUILDUP_IDX]

sig_adapt_viz = find_sigma_stop(
    model, x_viz, tgt_viz, tau=0.95, n_samples=32, n_iter=12)
lv_adapt_viz  = 2.0 * math.log(max(sig_adapt_viz, 1e-7))
lv_fixed_viz  = 2.0 * math.log(SIGMA_FINAL)

print(f'Image : {imagenet_labels[tgt_viz]}  '
      f'sigma_adapt={sig_adapt_viz:.3f}  idx={BUILDUP_IDX}')

# -- KL-IG² (fixed) trajectory -- from cached meta ---------------------------
_traj_key_mu  = 'traj_mu' if 'traj_mu' in meta_v else 'dist_traj_mu'
_traj_key_lv  = 'traj_lv' if 'traj_lv' in meta_v else 'dist_traj_lv'
traj_mu_fixed = [t.to(DEVICE) for t in meta_v[_traj_key_mu]]
traj_lv_fixed = [t.to(DEVICE) for t in meta_v[_traj_key_lv]]
K_fixed = len(traj_mu_fixed) - 1
print(f'KL-IG² (fixed)    path length: {K_fixed} steps')

# -- KL-IG² (adaptive) trajectory -- fresh descent ---------------------------
print('Building KL-IG² (adaptive) trajectory ...', end=' ', flush=True)
x_cf_viz = pick_cf_image(BUILDUP_IDX).squeeze(0).to(DEVICE)
_r_adapt_viz = KLIGSquared(
    model, phi, x_cf_viz,
    T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
    n_mc_path=N_MC_DESCENT, n_mc_grad=N_MC_BUILDUP,
    sigma_start=sig_adapt_viz, loss_stop=LOSS_STOP,
    lv_floor=2.0 * math.log(max(sig_adapt_viz, 1e-7)),
    lv_ceil=LV_CEIL, mu_min=MU_MIN, mu_max=MU_MAX,
    clamp_samples=True, device=DEVICE,
).attribute(x1_viz, target=tgt_viz)
traj_mu_adapt = _r_adapt_viz.traj_mu
traj_lv_adapt = _r_adapt_viz.traj_lv
K_adapt = len(traj_mu_adapt) - 1
print(f'done ({K_adapt} steps)')

# -- Helpers ------------------------------------------------------------------
def _to_rgb(t_chw: torch.Tensor) -> np.ndarray:
    return denormalize(t_chw.cpu()).clamp(0, 1).permute(1, 2, 0).numpy()


def get_baseline_image(method: str) -> np.ndarray:
    """The actual baseline each method integrates *from*."""
    if method in ('KLIG-Adaptive', 'KL-IG (linear)'):
        return _to_rgb(torch.zeros_like(x1_viz))   # zero baseline
    if method in ('KL-IG²', 'KL-IG² (adaptive)'):
        return _to_rgb(x_cf_viz)                    # actual CF image
    raise ValueError(method)


def get_baseline_label(method: str) -> str:
    if method in ('KLIG-Adaptive', 'KL-IG (linear)'):
        return 'Baseline\n(zero)'
    return 'Baseline\n(CF)'


# -- Partial attribution [baseline -> t_max] ----------------------------------
def _partial_klig_lin(x1, target, lv_scalar, t_max,
                       n_steps=N_STEPS_INT, n_mc=N_MC_BUILDUP):
    """LinearPath cumulative attr (zero baseline -> explicand)."""
    if t_max <= 0:
        return torch.zeros_like(x1)
    mu_f  = x1.detach()
    lv_f  = torch.full_like(mu_f, lv_scalar)
    attr  = torch.zeros_like(mu_f)
    saved = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)
    try:
        for k in range(n_steps):
            t_k = (k + 0.5) / n_steps
            if t_k > t_max:
                break
            mu_t  = (t_k * mu_f).detach().requires_grad_(True)
            lv_t  = (t_k * lv_f).detach().requires_grad_(True)
            eps   = torch.randn(n_mc, *mu_f.shape, device=DEVICE)
            xsamp = mu_t.unsqueeze(0) + (0.5 * lv_t).exp().unsqueeze(0) * eps
            g_mu, g_lv = torch.autograd.grad(
                model(xsamp)[:, target].mean(), [mu_t, lv_t])
            with torch.no_grad():
                attr.add_(g_mu * mu_f + g_lv * lv_f)
    finally:
        for p, s in zip(model.parameters(), saved): p.requires_grad_(s)
    return (attr / n_steps).detach()


def _partial_klig2_cf(target, traj_mu, traj_lv, t_max, n_mc=N_MC_BUILDUP):
    """KL-IG² cumulative attr_mu accumulated from CF end of path inward."""
    K = len(traj_mu) - 1
    if t_max <= 0 or K == 0:
        return torch.zeros_like(traj_mu[0])
    n_steps = max(1, min(int(round(t_max * K)), K))
    k_start = K - n_steps
    x_shape = traj_mu[0].shape
    attr_mu = torch.zeros_like(traj_mu[0])
    saved   = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)
    try:
        for k in range(k_start, K):
            mu_k  = traj_mu[k].to(DEVICE)
            lv_k  = traj_lv[k].to(DEVICE)
            dmu_k = mu_k - traj_mu[k + 1].to(DEVICE)
            mu_t  = mu_k.detach().requires_grad_(True)
            lv_t  = lv_k.detach().requires_grad_(True)
            eps   = torch.randn(n_mc, *x_shape, device=DEVICE)
            xsamp = mu_t.unsqueeze(0) + (0.5 * lv_t).exp().unsqueeze(0) * eps
            g_mu, _ = torch.autograd.grad(
                model(xsamp)[:, target].mean(), [mu_t, lv_t])
            with torch.no_grad():
                attr_mu.add_(g_mu * dmu_k)
    finally:
        for p, s in zip(model.parameters(), saved): p.requires_grad_(s)
    return attr_mu.detach()


def get_partial_attr(method: str, t_max: float) -> np.ndarray:
    if method == 'KLIG-Adaptive':
        a = _partial_klig_lin(x1_viz, tgt_viz, lv_adapt_viz, t_max)
    elif method == 'KL-IG (linear)':
        a = _partial_klig_lin(x1_viz, tgt_viz, lv_fixed_viz, t_max)
    elif method == 'KL-IG²':
        a = _partial_klig2_cf(tgt_viz, traj_mu_fixed, traj_lv_fixed, t_max)
    elif method == 'KL-IG² (adaptive)':
        a = _partial_klig2_cf(tgt_viz, traj_mu_adapt, traj_lv_adapt, t_max)
    else:
        raise ValueError(method)
    return absmax_collapse(a).clamp(min=0).cpu().numpy()

# -- Compute all (method x t) maps --------------------------------------------
print('Computing cumulative attribution maps ...')
buildup_maps = {m: [] for m in _BUILDUP_METHODS}
for m in tqdm(_BUILDUP_METHODS, desc='methods'):
    for t in T_STEPS:
        buildup_maps[m].append(get_partial_attr(m, t))
print('Done.')

# -- Render -------------------------------------------------------------------
# col 0 = original  |  col 1 = baseline image  |  cols 2..N_T+1 = cum attr
N_ROWS = len(_BUILDUP_METHODS)
N_COLS = 2 + N_T

METHOD_LABELS = {
    'KLIG-Adaptive':     f'KLIG-Adapt\nσ={sig_adapt_viz:.3f}',
    'KL-IG (linear)':    f'KL-IG Lin\nσ={SIGMA_FINAL}',
    'KL-IG²':            f'KL-IG²\nσ={SIGMA_FINAL}',
    'KL-IG² (adaptive)': f'KL-IG²-Adapt\nσ={sig_adapt_viz:.3f}',
}

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(2.15 * N_COLS, 2.4 * N_ROWS),
    facecolor='white',
    gridspec_kw={'wspace': 0.06, 'hspace': 0.12},
)
if N_ROWS == 1:
    axes = axes[np.newaxis, :]

img_orig = _to_rgb(x1_viz)

for mi, m in enumerate(_BUILDUP_METHODS):
    full_a = buildup_maps[m][-1]
    vmax = max(float(np.percentile(full_a, 99)), 1e-12)
    col = COLORS.get(m, 'black')

    # col 0: original image with row label
    ax0 = axes[mi, 0]
    ax0.imshow(img_orig)
    ax0.set_xticks([]); ax0.set_yticks([])
    for sp in ax0.spines.values(): sp.set_visible(False)
    ax0.set_ylabel(METHOD_LABELS[m], fontsize=8, color=col, fontweight='bold',
                   rotation=0, labelpad=70, va='center')
    if mi == 0:
        ax0.set_title('Original', fontsize=9, fontweight='bold')

    # col 1: baseline image (the actual integration starting point)
    ax_bl = axes[mi, 1]
    ax_bl.imshow(get_baseline_image(m))
    ax_bl.set_xticks([]); ax_bl.set_yticks([])
    for sp in ax_bl.spines.values(): sp.set_visible(False)
    if mi == 0:
        ax_bl.set_title('Baseline', fontsize=9, fontweight='bold')

    # cols 2..N_T+1: cumulative attribution heatmaps
    is_klig2 = m in ('KL-IG²', 'KL-IG² (adaptive)')
    for ci, t in enumerate(T_STEPS):
        ax_attr = axes[mi, 2 + ci]
        ax_attr.imshow(buildup_maps[m][ci], cmap='cividis', vmin=0, vmax=vmax)
        ax_attr.set_xticks([]); ax_attr.set_yticks([])
        for sp in ax_attr.spines.values(): sp.set_visible(False)
        if mi == 0:
            base = 'CF' if is_klig2 else '0'
            ax_attr.set_title(f'attr[{base}→{t:.2f}]', fontsize=8, fontweight='bold')

fig.suptitle(
    f'Path Attribution Build-up -- {imagenet_labels[tgt_viz]}\n'
    'KL-IG²: counterfactual baseline → explicand   |   KLIG: zero baseline → explicand',
    fontsize=11, fontweight='bold', y=1.01,
)
plt.savefig('path_buildup_klig_klig2_attr_only.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved path_buildup_klig_klig2_attr_only.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CS_latent — class-sensitivity in latent space
# T_yk = x * |A_yk|/max|A_yk|  ;  CS = distance(phi(T_y1), phi(T_y2))
# Higher = the two class attributions produce more separable representations.
# Reuses scatter helpers (_attr_for_class_fast, _build_klig2, _build_gradpath_once,
# _pick_cf_scatter) — RUN THE CS-SCATTER CELL FIRST so those exist.
# ════════════════════════════════════════════════════════════════════════════
import timm, pandas as pd
import torch.nn.functional as F
from captum.attr import IntegratedGradients

N_CS_LATENT = 50          # bump to >=200 for the final number
_rng_csl    = np.random.default_rng(0)

# ── Encoders ────────────────────────────────────────────────────────────────
# 1) ResNet-50 layer4 (avgpool output, 2048) — task-model oracle
_res_feats = {}
_res_hook  = model.avgpool.register_forward_hook(
    lambda m, i, o: _res_feats.__setitem__('z', o.detach()))
def enc_resnet(t):
    with torch.no_grad():
        model(t.unsqueeze(0).to(DEVICE))
    return _res_feats['z'].flatten()                      # (2048,)

# 2) ViT-B/16 patch tokens (mean-pooled, 768)
_vit = timm.create_model('vit_base_patch16_224', pretrained=True).to(DEVICE).eval()
def enc_vit(t):
    with torch.no_grad():
        f = _vit.forward_features(t.unsqueeze(0).to(DEVICE))   # (1, 1+N, 768)
    return f[0, 1:].mean(0)                                # exclude CLS -> (768,)

# 3) CLIP image encoder (512) — baseline oracle (what CASE uses implicitly)
def enc_clip(t):
    with torch.no_grad():
        vo = _clip_mdl.vision_model(pixel_values=t.unsqueeze(0).to(DEVICE))
        z  = _clip_mdl.visual_projection(vo.pooler_output)    # (1,512)
    return z[0]                                            # (512,)

ENCODERS = {'ResNet': enc_resnet, 'ViT': enc_vit, 'CLIP': enc_clip}

# ── IG baseline (captum, zero baseline) ──────────────────────────────────────
_ig = IntegratedGradients(model)
def _ig_attr(x1, cls):
    a = _ig.attribute(x1.unsqueeze(0), target=int(cls),
                      baselines=torch.zeros_like(x1).unsqueeze(0), n_steps=32)
    return absmax_collapse(a.squeeze(0)).detach().cpu().numpy()   # (H,W) signed

# ── CS_latent core ───────────────────────────────────────────────────────────
def cs_latent_score(x1, attr_y1, attr_y2, enc):
    # signed maps in, abs only for the mask magnitude; per-image normalize to [0,1]
    m1 = torch.from_numpy(np.abs(attr_y1)).float().to(DEVICE); m1 = m1/(m1.max()+1e-8)
    m2 = torch.from_numpy(np.abs(attr_y2)).float().to(DEVICE); m2 = m2/(m2.max()+1e-8)
    T1 = x1 * m1.unsqueeze(0)                              # (3,H,W) broadcast
    T2 = x1 * m2.unsqueeze(0)
    z1, z2 = enc(T1), enc(T2)
    cos = 1.0 - F.cosine_similarity(z1.unsqueeze(0), z2.unsqueeze(0)).item()
    l2  = torch.norm(z1 - z2).item()
    return cos, l2

# ── Methods → per-class attribution (signed (H,W)) ───────────────────────────
CSL_METHODS = ['KL-IG² (adaptive)', 'KL-IG²', 'KL-IG (linear)',
               'KLIG-Adaptive', 'IG', 'Random']

def _attr_map(method, x1, cls, H, W, ctx):
    if method == 'KLIG-Adaptive':
        return _attr_for_class_fast('KLIG-Adaptive', x1, cls,
                                    sig_adapt=ctx['sig'](cls)).reshape(H, W)
    if method == 'KL-IG (linear)':
        return _attr_for_class_fast('KL-IG (linear)', x1, cls).reshape(H, W)
    if method == 'KL-IG²':                                  # fixed-sigma
        return _attr_for_class_fast('KL-IG²', x1, cls,
                                    klig2_fixed=ctx['k2f'],
                                    path_fixed=ctx['pf']).reshape(H, W)
    if method == 'KL-IG² (adaptive)':
        k2, pth, sig = ctx['adapt'](cls)
        return _attr_for_class_fast('KL-IG² (adaptive)', x1, cls,
                                    klig2_adapt=k2, path_adapt=pth,
                                    sig_adapt=sig).reshape(H, W)
    if method == 'IG':
        return _ig_attr(x1, cls)
    if method == 'Random':
        return _rng_csl.uniform(-1, 1, size=(H, W)).astype('float32')
    raise ValueError(method)

# ── Run ──────────────────────────────────────────────────────────────────────
Y2_MIN_PROB = 0.10   # CF = next predicted class, only if its prob exceeds this
import random
_cands = [d for d in multi_imgs
          if d.get('high_cls') and len(d['high_cls']) >= 2
          and d['high_probs'][1] > Y2_MIN_PROB]
random.seed(0)                              # pinned -> reproducible sample
random.shuffle(_cands)
# prefer unique top-1 classes for maximum variety, then pad to N
_pool, _seen_cls = [], set()
for d in _cands:
    if d['high_cls'][0] not in _seen_cls:
        _pool.append(d); _seen_cls.add(d['high_cls'][0])
    if len(_pool) >= N_CS_LATENT: break
if len(_pool) < N_CS_LATENT:
    _ids = {id(d) for d in _pool}
    for d in _cands:
        if len(_pool) >= N_CS_LATENT: break
        if id(d) not in _ids: _pool.append(d)
print(f'CS_latent over {len(_pool)} images (y2 prob > {Y2_MIN_PROB}, '
      f'{len(set(d["high_cls"][0] for d in _pool))} distinct top-1 classes) '
      f'x {len(CSL_METHODS)} methods x {len(ENCODERS)} encoders')

# ── CF pool for THIS set's y2 classes (best-by-probability, guaranteed real CF) ─
_csl_cache = CACHE_DIR / 'klig2_cf_csl_pool.pkl'
_need = {int(d['high_cls'][1]) for d in _pool}
if not FORCE_RECOMPUTE and _csl_cache.exists():
    _cf_cpu = pickle.load(open(_csl_cache, 'rb'))
    if len(set(_cf_cpu) & _need) < len(_need): _csl_cache.unlink()
if FORCE_RECOMPUTE or not _csl_cache.exists():
    _cf_cpu = {}
    # seed from already-loaded multi_imgs (top-1==c -> real image of class c, no download)
    for _d2 in multi_imgs:
        _c0 = int(_d2['high_cls'][0])
        if _c0 in _need and _c0 not in _cf_cpu:
            _xx = _d2['x']; _xx = _xx.squeeze(0) if _xx.dim() == 4 else _xx
            _cf_cpu[_c0] = _xx.cpu()
    # then seed from cf_pool (already built)
    for _c in _need:
        if _c not in _cf_cpu and 'cf_pool' in globals() and _c in cf_pool:
            _cf_cpu[_c] = cf_pool[_c].cpu()
    _still = _need - set(_cf_cpu)
    print(f'  seeded {len(_cf_cpu)}/{len(_need)} from memory; streaming for {len(_still)}')
    if _still:
        from datasets import load_dataset as _hf
        _s = _hf('evanarlian/imagenet_1k_resized_256', split='val',
                 streaming=True).shuffle(seed=13, buffer_size=500)
        _best, _lock, _sc = {c: (-1.0, None) for c in _still}, set(), 0
        _pb = tqdm(total=len(_still), desc='CS_latent CF pool')
        for item in _s:
            _sc += 1
            if len(_lock) >= len(_still) or _sc >= 8000: break
            im = item['image']
            if im.mode != 'RGB': im = im.convert('RGB')
            xx = preprocess(im).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                pr = model(xx).softmax(-1)[0].cpu()
            xc = xx.cpu()
            for c in _still:
                if c in _lock: continue
                if float(pr[c]) > _best[c][0]:
                    _best[c] = (float(pr[c]), xc)
                    if pr[c] >= 0.30: _lock.add(c); _pb.update(1)
        _pb.close()
        for c in _still:
            if _best[c][1] is not None: _cf_cpu[c] = _best[c][1]
    pickle.dump(_cf_cpu, open(_csl_cache, 'wb'))
cf_csl = {c: v.to(DEVICE) for c, v in _cf_cpu.items()}
print(f'CS_latent CF pool: {len(cf_csl)}/{len(_need)} y2 classes')

rows = []
for d in tqdm(_pool, desc='CS_latent'):
    x1   = d['x'].squeeze(0).to(DEVICE)
    H, W = x1.shape[1], x1.shape[2]
    y1, y2 = int(d['high_cls'][0]), int(d['high_cls'][1])
    img_id = d['idx']

    # per-image KL-IG² context (CF + fixed path + per-class adaptive path)
    x_cf = cf_csl.get(y2)
    if x_cf is None:        # no real CF for this y2 -> skip image
        continue
    if x_cf.dim() == 4: x_cf = x_cf.squeeze(0)
    x_cf = x_cf.to(DEVICE)
    k2f = _build_klig2(x_cf, SIGMA_FINAL, LV_FLOOR)
    pf  = _build_gradpath_once(k2f, x1)
    _sig_cache, _ad_cache = {}, {}
    def _sig(cls):
        if cls not in _sig_cache:
            _sig_cache[cls] = find_sigma_stop(model, x1, int(cls), tau=0.95,
                                              n_samples=32, n_iter=12)
        return _sig_cache[cls]
    def _adapt(cls):
        if cls not in _ad_cache:
            s = _sig(cls); k2 = _build_klig2(x_cf, s, 2*math.log(s))
            _ad_cache[cls] = (k2, _build_gradpath_once(k2, x1), s)
        return _ad_cache[cls]
    ctx = {'k2f': k2f, 'pf': pf, 'sig': _sig, 'adapt': _adapt}

    for m in CSL_METHODS:
        a1 = _attr_map(m, x1, y1, H, W, ctx)
        a2 = _attr_map(m, x1, y2, H, W, ctx)
        for enc_name, enc in ENCODERS.items():
            cos, l2 = cs_latent_score(x1, a1, a2, enc)
            rows.append({'image_id': img_id, 'method': m, 'encoder': enc_name,
                         'cs_cosine': cos, 'cs_l2': l2, 'y1': y1, 'y2': y2})

_res_hook.remove()
df_csl = pd.DataFrame(rows)
df_csl.to_csv('cs_latent_results.csv', index=False)
print('saved cs_latent_results.csv  (', len(df_csl), 'rows )')

# ── Summary table: mean +/- std cosine per method x encoder ──────────────────
print('\nCS_latent (cosine) — mean +/- std  [higher = more class-sensitive]')
piv = (df_csl.groupby(['method', 'encoder'])['cs_cosine']
       .agg(['mean', 'std']).reset_index())
tab = piv.pivot(index='method', columns='encoder', values='mean').reindex(CSL_METHODS)
print(tab[['ResNet', 'ViT', 'CLIP']].round(4).to_string())
print('\n(full per-image data in cs_latent_results.csv; cs_l2 also stored)')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CS_latent — 2D latent-space visualization (paired phi(T_y1) / phi(T_y2), ViT)
# Two panels: KL-IG² (adaptive) vs KL-IG (linear). One shared projection (fit once).
# filled circle = y1, open circle = y2, line = the pair. Longer line => more
# class-separable in latent space.
# PREREQ: run the CS_latent cell first (needs _pool, cf_csl, _attr_map, enc_vit,
#         _build_klig2, _build_gradpath_once, find_sigma_stop).
# ════════════════════════════════════════════════════════════════════════════
PROJ_METHOD = 'UMAP'   # 'UMAP' = nice clusters | 'PCA' = line length ~ true CS scale
from sklearn.decomposition import PCA
if PROJ_METHOD == 'UMAP':
    try:
        import umap
    except Exception:
        print('umap unavailable -> falling back to PCA'); PROJ_METHOD = 'PCA'
_PROJ = PROJ_METHOD

VIZ2D_METHODS = ['KL-IG² (adaptive)', 'KLIG-Adaptive']   # both adaptive

recs = []   # (method, z_y1, z_y2, cs_cosine)
for d in tqdm(_pool, desc='2D latent viz'):
    x1   = d['x'].squeeze(0).to(DEVICE)
    H, W = x1.shape[1], x1.shape[2]
    y1, y2 = int(d['high_cls'][0]), int(d['high_cls'][1])
    x_cf = cf_csl.get(y2)
    if x_cf is None:
        continue
    x_cf = (x_cf.squeeze(0) if x_cf.dim() == 4 else x_cf).to(DEVICE)

    k2f = _build_klig2(x_cf, SIGMA_FINAL, LV_FLOOR)
    pf  = _build_gradpath_once(k2f, x1)
    _sc, _ac = {}, {}
    def _sig(cls):
        if cls not in _sc:
            _sc[cls] = find_sigma_stop(model, x1, int(cls), tau=0.95,
                                       n_samples=32, n_iter=12)
        return _sc[cls]
    def _adapt(cls):
        if cls not in _ac:
            s = _sig(cls); k2 = _build_klig2(x_cf, s, 2*math.log(s))
            _ac[cls] = (k2, _build_gradpath_once(k2, x1), s)
        return _ac[cls]
    ctx = {'k2f': k2f, 'pf': pf, 'sig': _sig, 'adapt': _adapt}

    for m in VIZ2D_METHODS:
        a1 = _attr_map(m, x1, y1, H, W, ctx)
        a2 = _attr_map(m, x1, y2, H, W, ctx)
        m1 = torch.from_numpy(np.abs(a1)).float().to(DEVICE); m1 = m1/(m1.max()+1e-8)
        m2 = torch.from_numpy(np.abs(a2)).float().to(DEVICE); m2 = m2/(m2.max()+1e-8)
        z1 = enc_vit(x1 * m1.unsqueeze(0)).cpu().numpy()
        z2 = enc_vit(x1 * m2.unsqueeze(0)).cpu().numpy()
        cs = 1.0 - float(z1 @ z2 / (np.linalg.norm(z1)*np.linalg.norm(z2) + 1e-9))
        recs.append((m, z1, z2, cs))

# ── one shared projection over ALL z (both methods, both y1/y2) ───────────────
n  = len(recs)
Z  = np.stack([r[1] for r in recs] + [r[2] for r in recs])    # (2n, d)
if _PROJ == 'UMAP':
    P = umap.UMAP(n_components=2, random_state=0,
                  n_neighbors=15, min_dist=0.1).fit_transform(Z)
else:
    P = PCA(n_components=2, random_state=0).fit_transform(Z)
P1, P2 = P[:n], P[n:]                                          # y1 pts, y2 pts

# ── plot: two panels, shared axes ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')
xlim = (P[:,0].min()-1, P[:,0].max()+1)
ylim = (P[:,1].min()-1, P[:,1].max()+1)
for ax, m in zip(axes, VIZ2D_METHODS):
    col = COLORS.get(m, 'gray')
    idx = [i for i, r in enumerate(recs) if r[0] == m]
    for i in idx:
        ax.plot([P1[i,0], P2[i,0]], [P1[i,1], P2[i,1]],
                '-', color=col, alpha=0.45, lw=1, zorder=1)
        ax.scatter(P1[i,0], P1[i,1], facecolor=col, edgecolor='black',
                   s=45, lw=0.6, zorder=3)                      # y1 filled
        ax.scatter(P2[i,0], P2[i,1], facecolor='white', edgecolor=col,
                   s=45, lw=1.2, zorder=3)                      # y2 open
    sep = np.mean([np.linalg.norm(P1[i]-P2[i]) for i in idx]) if idx else 0.0
    cs_mean = np.mean([recs[i][3] for i in idx]) if idx else 0.0
    ax.set_title(f'{m}\nmean 2D separation = {sep:.2f}   |   CS_latent = {cs_mean:.3f}',
                 fontsize=11, fontweight='bold', color=col)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xticks([]); ax.set_yticks([])

from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='gray', markerfacecolor='gray',
              markeredgecolor='black', lw=0, label='y1 (target)'),
       Line2D([0],[0], marker='o', color='gray', markerfacecolor='white',
              markeredgecolor='gray', lw=0, label='y2 (counterfactual)')]
axes[0].legend(handles=leg, loc='upper left', fontsize=9, framealpha=0.9)

fig.suptitle(f'CS_latent — {_PROJ} of ViT phi(T_y1) vs phi(T_y2)   '
             f'(longer line = more class-separable)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cs_latent_2d.png', dpi=160, bbox_inches='tight')
plt.show()
print(f'saved cs_latent_2d.png  ({_PROJ}, {n} pairs/method)')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CS_latent — Option B: PCA of DELTA vectors  delta = phi(T_y1) - phi(T_y2)  (ViT)
# One delta per (image, method). PCA fit once over ALL deltas. Color by method.
# KL-IG² deltas spread far from origin (class-specific directions);
# IG / Random deltas cluster near 0.
# PREREQ: run the CS_latent cell first (_pool, cf_csl, _attr_map, enc_vit, ...).
# ════════════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA

DELTA_METHODS = ['KL-IG² (adaptive)', 'KLIG-Adaptive', 'IG', 'Random']

deltas, dmeth = [], []
for d in tqdm(_pool, desc='delta vectors'):
    x1   = d['x'].squeeze(0).to(DEVICE)
    H, W = x1.shape[1], x1.shape[2]
    y1, y2 = int(d['high_cls'][0]), int(d['high_cls'][1])
    x_cf = cf_csl.get(y2)
    if x_cf is None:
        continue
    x_cf = (x_cf.squeeze(0) if x_cf.dim() == 4 else x_cf).to(DEVICE)
    k2f = _build_klig2(x_cf, SIGMA_FINAL, LV_FLOOR)
    pf  = _build_gradpath_once(k2f, x1)
    _sc, _ac = {}, {}
    def _sig(c):
        if c not in _sc:
            _sc[c] = find_sigma_stop(model, x1, int(c), tau=0.95, n_samples=32, n_iter=12)
        return _sc[c]
    def _adapt(c):
        if c not in _ac:
            s = _sig(c); k2 = _build_klig2(x_cf, s, 2*math.log(s))
            _ac[c] = (k2, _build_gradpath_once(k2, x1), s)
        return _ac[c]
    ctx = {'k2f': k2f, 'pf': pf, 'sig': _sig, 'adapt': _adapt}
    for m in DELTA_METHODS:
        a1 = _attr_map(m, x1, y1, H, W, ctx)
        a2 = _attr_map(m, x1, y2, H, W, ctx)
        m1 = torch.from_numpy(np.abs(a1)).float().to(DEVICE); m1 = m1/(m1.max()+1e-8)
        m2 = torch.from_numpy(np.abs(a2)).float().to(DEVICE); m2 = m2/(m2.max()+1e-8)
        z1 = enc_vit(x1 * m1.unsqueeze(0)).cpu().numpy()
        z2 = enc_vit(x1 * m2.unsqueeze(0)).cpu().numpy()
        deltas.append(z1 - z2); dmeth.append(m)

D     = np.stack(deltas)
dmeth = np.array(dmeth)
P     = PCA(n_components=2, random_state=0).fit_transform(D)   # fit once on all deltas

# ── scatter ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7), facecolor='white')
ax.axhline(0, color='gray', lw=0.6, zorder=0)
ax.axvline(0, color='gray', lw=0.6, zorder=0)
for m in DELTA_METHODS:
    mask = dmeth == m
    ax.scatter(P[mask, 0], P[mask, 1], s=38, alpha=0.7,
               color=COLORS.get(m, 'gray'), edgecolor='white', lw=0.4, label=m)
ax.scatter([0], [0], marker='+', s=220, color='black', lw=2, zorder=5)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('CS_latent delta vectors  phi(T_y1) - phi(T_y2)   (ViT, PCA)\n'
             'far from origin = stronger class-specific direction',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig('cs_latent_delta_pca.png', dpi=160, bbox_inches='tight')
plt.show()

# ── spread stats (the quantitative story) ─────────────────────────────────────
print(f"\n{'method':22s} {'mean||delta||':>13s} {'std||delta||':>12s} {'mean PCA radius':>16s}")
print('-' * 66)
for m in DELTA_METHODS:
    mask = dmeth == m
    norms = np.linalg.norm(D[mask], axis=1)
    rad   = np.linalg.norm(P[mask], axis=1)
    print(f'{m:22s} {norms.mean():13.3f} {norms.std():12.3f} {rad.mean():16.3f}')
print('\n(mean||delta|| in full ViT space is the faithful spread measure; '
      'PCA radius is its 2D shadow)')
print('saved cs_latent_delta_pca.png')


# Class Sensitivity — Modular (all 11 methods, N images)

Runs the three class-sensitivity metrics over **11 methods** (6 baselines + ExpGrad + 4 KLIG)
via `klig_methods.py`, `case_metric.py`, `class_sens_cosine_clip.py`, `klig_cs_lishi.py`.
Only **KL-IG²** uses the counterfactual baseline; the others use their native baselines.

**Run before this section:** cells 1-8 (model + attribution) and **Class Sens — Step 1**
(defines `multi_imgs`). Step 2/3 are NOT required — the setup cell builds its own CLIP +
counterfactual pool. Set `MAX_CS_IMAGES` (default 100) in the setup cell.


In [ ]:
# ── Modular class-sensitivity setup: 11 methods on N images (proper KL-IG² CFs) ──
# Self-contained: needs only cells 1-8 (model + attribution) and "Class Sens — Step 1"
# (which defines `multi_imgs`).  Does NOT need Step 2/3.
import importlib
import klig_methods, case_metric, class_sens_cosine_clip, klig_cs_lishi
for _m in (klig_methods, case_metric, class_sens_cosine_clip, klig_cs_lishi):
    importlib.reload(_m)
import klig_methods as KM
from case_metric import run_case, case_summary_table
from class_sens_cosine_clip import run_cs_cosine_clip, plot_cs_scatter, cs_rho, build_clip
from klig_cs_lishi import run_cs_lishi, write_outputs

MAX_CS_IMAGES = 100            # <-- run the modular metrics on this many images

# faster KL-IG²/IG settings so a 100-image sweep is tractable (the "scatter" config)
KM.N_STEPS, KM.N_SAMPLES = 25, 3
KM.IG_STEPS = 25
KM.SG_SAMPLES, KM.EG_SAMPLES = 25, 25
KM.T_DESC, KM.N_MC_DESC, KM.N_MC_GRAD = 25, 8, 3

phi_km      = KM.make_phi(model)
METHODS_ALL = KM.METHODS
print('methods (%d):' % len(METHODS_ALL), METHODS_ALL)

# 1) image pool = multi-class val images (from "Class Sens — Step 1")
_src = list(multi_imgs)[:MAX_CS_IMAGES]
print(f'using {len(_src)}/{len(multi_imgs)} multi-class images (target {MAX_CS_IMAGES})')

# 2) CLIP text embeddings for d_sem (built once, offline)
clip_km = build_clip(imagenet_labels, DEVICE)

# 3) counterfactual (Top-2-class image) per image — needed by KL-IG² only
need_y2 = {int(d['high_cls'][1]) for d in _src if len(d['high_cls']) > 1}
cf_by_cls = {}
for _name in ('cf_scatter_pool', 'cf_pool'):
    P = globals().get(_name, {}) or {}
    for c in list(need_y2):
        if c in P and c not in cf_by_cls:
            cf_by_cls[c] = P[c]
missing = sorted(need_y2 - set(cf_by_cls))
print(f'CF pool: have {len(cf_by_cls)}/{len(need_y2)} y2 classes; building {len(missing)} more (val stream)...')
if missing:
    from datasets import load_dataset as _hf
    _s = _hf('evanarlian/imagenet_1k_resized_256', split='val', streaming=True).shuffle(seed=11, buffer_size=5000)
    best = {c: (-1.0, None) for c in missing}; locked = set(); scanned = 0
    for item in tqdm(_s, desc='CF pool (val)'):
        scanned += 1
        if len(locked) >= len(missing) or scanned >= 20000:
            break
        im = item['image']
        if im.mode != 'RGB': im = im.convert('RGB')
        xx = preprocess(im).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = model(xx).softmax(-1)[0].cpu()
        for c in missing:
            if c in locked: continue
            p = float(probs[c])
            if p > best[c][0]:
                best[c] = (p, xx.squeeze(0))
                if p >= 0.30: locked.add(c)
    for c in missing:
        if best[c][1] is not None:
            cf_by_cls[c] = best[c][1]
    print(f'  built CFs for {sum(best[c][1] is not None for c in missing)}/{len(missing)} (scanned {scanned})')

def _cf_for(d):
    c = int(d['high_cls'][1]) if len(d['high_cls']) > 1 else int(d['high_cls'][0])
    cf = cf_by_cls.get(c) or (next(iter(cf_by_cls.values())) if cf_by_cls else None)
    if cf is not None and hasattr(cf, 'dim') and cf.dim() == 4:
        cf = cf.squeeze(0)
    return cf.to(DEVICE) if cf is not None else None

cs_images = [{'x': d['x'], 'high_cls': d['high_cls'], 'cf': _cf_for(d)} for d in _src]
print('class-sensitivity images ready:', len(cs_images))


In [ ]:
# ── Cosine d_attr vs CLIP d_sem — all 11 methods ────────────────────────────
scatter_km = run_cs_cosine_clip(cs_images, model, METHODS_ALL, clip_km, phi=phi_km)
plot_cs_scatter(scatter_km, KM.COLORS, out_png='cs_cosine_clip_val.png')
print('Spearman rho (d_sem vs d_attr):')
for m, (rho, p, n) in cs_rho(scatter_km).items():
    print(f'  {m:<20} rho={rho:+.3f}  p={p:.3g}  n={n}')


In [ ]:
# ── CASE feature-agreement — all 11 methods ─────────────────────────────────
case_km = run_case(cs_images, model, METHODS_ALL, phi=phi_km)
case_summary_table(case_km, out_png='case_summary_val.png')
for m, r in sorted(case_km.items(), key=lambda kv: kv[1]['median']):
    print(f"  {m:<20} median_FA={r['median']:.3f}  p={r['wilcoxon_p']:.3g}  class-distinct={r['reject_H0']}")


In [ ]:
# ── Li & Shi pixel-space CS — all 11 methods (high_low + top1_top2) ──────────
lishi_rows, lishi_summary = run_cs_lishi(cs_images, model, METHODS_ALL, phi=phi_km)
write_outputs(lishi_rows, lishi_summary, csv_path='cs_lishi_val.csv',
              summary_csv='cs_lishi_summary_val.csv', png='cs_lishi_summary_val.png')
for pt in ('high_low', 'top1_top2'):
    print(f'[{pt}] rank ascending by mean_cs:')
    blk = sorted([s for s in lishi_summary if s['class_pair_type']==pt],
                 key=lambda s: (9 if s['mean_cs']!=s['mean_cs'] else s['mean_cs']))
    for i, s in enumerate(blk):
        print(f"  {i+1:>2}. {s['method']:<20} mean={s['mean_cs']:+.3f}  n_deg={s['n_degenerate']}  n={s['n']}")
